
# Проверка систематик SBF для трёх галактик

Ноутбук идёт **по этапам**. Каждая вычислительная ячейка сначала выполняет
один и тот же этап для NGC 3379, NGC 1380 и NGC 4486, а затем показывает
общую таблицу или общий набор графиков.

Пользовательские результаты хранятся в обычных `DataFrame` с колонкой
`galaxy`: `input_summary_df`, `winsor_thresholds_df`, `noise_tiles_df`,
`power_spectra_df`, `fit_summary_df`, `psf_comparison_df` и
`pilot_summary_df`. Внутренняя колонка `_state` таблицы `galaxies` нужна
только для больших FITS-массивов и не выводится на экран.

Научная логика production-ветви не меняется. Модель коррелированного шума,
порог винзорирования и размер PSF остаются проверками чувствительности.


In [1]:
import json
import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from types import SimpleNamespace
from importlib.metadata import version as package_version
from IPython.display import display
from matplotlib.patches import Patch
from astropy.io import fits
from astropy.stats import sigma_clipped_stats
from scipy import ndimage
from scipy.fft import fft2, fftfreq, set_workers

Matplotlib is building the font cache; this may take a moment.


## 1. Выборка и параметры расчёта

Таблица ниже — единая точка выбора галактик и их визуальных оценок качества.

In [2]:
galaxies = pd.DataFrame({
    "galaxy": ["NGC 3379", "NGC 1380", "NGC 4486"],
    "visual_score": [10.0, 7.0, 3.0],
})
GALAXIES = galaxies["galaxy"].tolist()

K_MIN, K_MAX, K_BINS = 0.04, 0.25, 80
E_REALIZATIONS = N_REALIZATIONS = 64
FFT_WORKERS = -1
RANDOM_SEED = 1489

TILE_SIZE = 512
TILE_MARGIN = 96
MIN_TILE_VALID_FRACTION = 0.80
RELAXED_TILE_VALID_FRACTION = 0.55
MAX_NOISE_TILES = 4
REPRO_PIXEL_TOL = 1e-5
REPRO_POWER_REL_TOL = 0.005
REPRO_MAG_TOL = 0.005

WINSOR_VARIANTS = [
    ("raw", None),
    ("winsor_3.0", 3.0),
    ("winsor_3.5", 3.5),
    ("winsor_4.0", 4.0),
]

PSF_NLAMBDA = 7
PSF_FFT_OVERSAMPLE = 4
PSF_DETECTOR_OVERSAMPLE = 1
PSF_ADD_DISTORTION = True
PSF_SOURCE_ARGUMENT = "None"
PSF_SOURCE_SPECTRUM = "STPSF default 5700 K sunlike star"

project_root = Path.cwd().resolve()
if project_root.name == "code":
    project_root = project_root.parent
if not (project_root / "runs" / "sbf2_go3055").is_dir():
    raise RuntimeError("Запустите ноутбук из корня проекта или из папки code")

pilot_root = project_root / "runs" / "sbf2_systematics" / "pilot"
pilot_table_dir = pilot_root / "tables"
pilot_figure_dir = pilot_root / "figures"
pilot_table_dir.mkdir(parents=True, exist_ok=True)
pilot_figure_dir.mkdir(parents=True, exist_ok=True)

display(galaxies)


,galaxy,visual_score
0,NGC 3379,10.0
1,NGC 1380,7.0
2,NGC 4486,3.0


## 2. Повторно используемые численные операции

Эти три функции вызываются для каждой галактики; сами они ничего не выводят.

In [3]:
def radial_mean_sem(power, radial_plan):
    values = np.asarray(power, dtype=float).ravel()
    selected = radial_plan["valid"] & np.isfinite(values)
    ids = radial_plan["ids"][selected]
    values = values[selected]
    n_bins = radial_plan["n_bins"]

    count = np.bincount(ids, minlength=n_bins).astype(int)
    total = np.bincount(ids, weights=values, minlength=n_bins)
    total2 = np.bincount(ids, weights=values**2, minlength=n_bins)

    mean = np.full(n_bins, np.nan)
    sem = np.full(n_bins, np.nan)
    enough = count >= 10
    mean[enough] = total[enough] / count[enough]

    variance = np.zeros(n_bins)
    variance[enough] = (
        total2[enough] - total[enough]**2 / count[enough]
    ) / np.maximum(count[enough] - 1, 1)
    variance = np.maximum(variance, 0)
    sem[enough] = np.sqrt(variance[enough] / count[enough])
    return mean, sem, count


def monte_carlo_template(
    window, radial_plan, fourier_filter, n_realizations, seed, amplitude=None
):
    rng = np.random.default_rng(seed)
    n_use = int(window.sum())
    power_sum = np.zeros(window.shape, dtype=float)

    for _ in range(n_realizations):
        white = rng.normal(size=window.shape)
        field = np.real(np.fft.ifft2(fft2(white) * fourier_filter))
        if amplitude is not None:
            field *= amplitude
        sampled = np.zeros_like(field)
        sampled[window] = field[window]
        sampled[window] -= np.mean(sampled[window])
        power_sum += np.abs(fft2(sampled))**2 / n_use

    profile, _, _ = radial_mean_sem(
        power_sum / n_realizations,
        radial_plan,
    )
    return profile


def weighted_fit(y, y_error, first_template, second_template):
    design = np.column_stack([first_template, second_template])
    safe_error = np.maximum(y_error, 1e-12)
    weighted_design = design / safe_error[:, None]
    weighted_y = y / safe_error

    coefficients = np.linalg.lstsq(
        weighted_design,
        weighted_y,
        rcond=None,
    )[0]
    model_values = design @ coefficients
    residual = y - model_values
    chi2 = float(np.sum((residual / safe_error)**2))
    dof = max(len(y) - 2, 1)
    chi2_reduced = chi2 / dof

    normal = weighted_design.T @ weighted_design
    covariance = np.linalg.pinv(normal) * max(chi2_reduced, 1.0)
    condition_number = float(np.linalg.cond(weighted_design))

    covariance_scale = np.sqrt(covariance[0, 0] * covariance[1, 1])
    covariance_correlation = (
        float(covariance[0, 1] / covariance_scale)
        if covariance_scale > 0 else np.nan
    )
    template_correlation = (
        float(np.corrcoef(first_template, second_template)[0, 1])
        if np.std(second_template) > 0
        else np.nan
    )

    return {
        "P0": float(coefficients[0]),
        "noise_coefficient": float(coefficients[1]),
        "P0_sigma": float(np.sqrt(max(covariance[0, 0], 0))),
        "cov_P0_P0": float(covariance[0, 0]),
        "cov_P0_Pnoise": float(covariance[0, 1]),
        "cov_Pnoise_P0": float(covariance[1, 0]),
        "cov_Pnoise_Pnoise": float(covariance[1, 1]),
        "chi2": chi2,
        "chi2_reduced": chi2_reduced,
        "aicc": chi2 + 4.0 + 12.0 / max(len(y) - 3, 1),
        "condition_number": condition_number,
        "template_correlation": template_correlation,
        "covariance_correlation": covariance_correlation,
        "model_values": model_values,
        "standardized_residual": residual / safe_error,
    }


## 3. Загрузка входных данных

На экран выводится одна строка на галактику: готовность входов, число
пикселей в двух рабочих кольцах и размер ансамбля PSF. Полные пути,
времена файлов и технические идентификаторы сохраняются в CSV, но здесь
не показываются.


In [4]:
def load_galaxy(galaxy):
    GALAXY = galaxy

    validation_root = (
        project_root / "runs" / "sbf2_systematics"
        / GALAXY.replace(" ", "_")
    )
    table_dir = validation_root / "tables"
    figure_dir = validation_root / "figures"
    table_dir.mkdir(parents=True, exist_ok=True)
    figure_dir.mkdir(parents=True, exist_ok=True)

    galaxy_tag = GALAXY.lower().replace(" ", "_")
    result_path = (
        project_root / "runs" / "sbf2_go3055" / "batch"
        / f"{GALAXY.replace(' ', '_')}_result.json"
    )
    if not result_path.is_file():
        raise FileNotFoundError(result_path)

    result = json.loads(result_path.read_text())
    if result.get("status") != "ok":
        raise RuntimeError(f"{GALAXY}: batch status = {result.get('status')}")

    run_dir = Path(result["output_dir"])
    stem = result["stem"]

    signal_path = Path(result["signal_path"])
    model_path = Path(result["model_full_fits"])
    used_residual_path = Path(result["science_residual_fits"])
    measurement_path = Path(result["df_sbf_csv"])

    inner_path = Path(result["inner_usable_residual_fits"])
    outer_path = Path(result["outer_usable_residual_fits"])

    catalog_mask_path = run_dir / f"{stem}_sbf_catalog_mask_mcut.fits"
    psf_path = run_dir / f"{stem}_psf_129.fits"

    required_paths = [
        signal_path,
        model_path,
        used_residual_path,
        measurement_path,
        catalog_mask_path,
        psf_path,
        inner_path,
        outer_path,
    ]
    missing = [path for path in required_paths if not path.is_file()]
    if missing:
        raise FileNotFoundError("\n".join(str(path) for path in missing))

    with fits.open(signal_path, memmap=True) as hdul:
        pixel_area = float(hdul["SCI"].header["PIXAR_SR"]) / 2.350443e-11
        science = np.asarray(hdul["SCI"].data, dtype=float)
        error_map = np.array(hdul["ERR"].data, dtype=np.float32)
        weight_map = np.array(hdul["WHT"].data, dtype=np.float32)

    with fits.open(model_path, memmap=True) as hdul:
        model = np.asarray(hdul[0].data, dtype=float)

    catalog_mask = np.asarray(
        fits.getdata(catalog_mask_path, memmap=True), dtype=bool
    )

    saved_residual = np.asarray(
        fits.getdata(used_residual_path), dtype=np.float32
    ).copy()
    residual_raw = np.asarray(
        (science - float(result["signal_background_scalar"])) - model,
        dtype=np.float32,
    )
    del science

    reconstructed_science_mask = (
        (~catalog_mask)
        & np.isfinite(residual_raw)
        & np.isfinite(model)
        & (model > 0)
    )
    science_mask = reconstructed_science_mask
    residual_raw[~science_mask] = np.nan
    # saved_residual нужен ниже для строгой проверки ветки 3.5 sigma

    ring_masks = {
        "inner": np.isfinite(
            fits.getdata(inner_path, memmap=True)
        ) & science_mask,
        "outer": np.isfinite(
            fits.getdata(outer_path, memmap=True)
        ) & science_mask,
    }

    with fits.open(psf_path, memmap=True) as hdul:
        psf_ids = []
        psfs = []
        for index, hdu in enumerate(hdul[1:]):
            if hdu.data is None or np.ndim(hdu.data) != 2:
                continue
            psf = np.array(hdu.data, dtype=float)
            psf /= psf.sum()
            psfs.append(psf)
            psf_ids.append(hdu.header.get("PSFID", hdu.name or f"PSF{index:02d}"))

    if len(psfs) != 5:
        raise RuntimeError(f"Ожидались 5 PSF, найдено {len(psfs)}")

    measurements = pd.read_csv(measurement_path)
    main_measurements = measurements[
        measurements["region"].isin(["circular_inner_lit", "circular_outer_lit"])
        & np.isclose(measurements["kmin"], K_MIN)
        & np.isclose(measurements["kmax"], K_MAX)
        & measurements["status"].eq("ok")
    ].copy()
    if len(main_measurements) != 2:
        raise RuntimeError("Не найдены оба production-измерения в основном k-окне")

    region_name = {
        "inner": "circular_inner_lit",
        "outer": "circular_outer_lit",
    }
    pr_by_ring = {
        ring: float(
            main_measurements.loc[
                main_measurements["region"].eq(region_name[ring]), "Pr"
            ].iloc[0]
        )
        for ring in ring_masks
    }

    jy_per_pixel = 2.350443e-5 * pixel_area
    ab_zeropoint = float(-2.5 * np.log10(jy_per_pixel / 3631.0))

    input_rows = []
    for name, path in {
        "signal": signal_path,
        "model": model_path,
        "catalog_mask": catalog_mask_path,
        "production_residual": used_residual_path,
        "inner_ring": inner_path,
        "outer_ring": outer_path,
        "measurements": measurement_path,
        "psf_ensemble": psf_path,
    }.items():
        stat = path.stat()
        input_rows.append({
            "galaxy": GALAXY,
            "name": name,
            "file": path.name,
            "path": str(path),
            "size_bytes": stat.st_size,
            "mtime_ns": stat.st_mtime_ns,
        })

    input_manifest = pd.DataFrame(input_rows)
    input_manifest.to_csv(
        table_dir / f"{galaxy_tag}_noise_winsor_input_manifest.csv", index=False
    )
    return SimpleNamespace(
        galaxy=GALAXY,
        validation_root=validation_root,
        table_dir=table_dir,
        figure_dir=figure_dir,
        galaxy_tag=galaxy_tag,
        result=result,
        run_dir=run_dir,
        stem=stem,
        signal_path=signal_path,
        model=model,
        error_map=error_map,
        weight_map=weight_map,
        saved_residual=saved_residual,
        residual_raw=residual_raw,
        science_mask=science_mask,
        ring_masks=ring_masks,
        psf_ids=psf_ids,
        psfs=psfs,
        main_measurements=main_measurements,
        region_name=region_name,
        pr_by_ring=pr_by_ring,
        ab_zeropoint=ab_zeropoint,
        input_manifest=input_manifest,
        load_row={
            "galaxy": GALAXY,
            "ready": True,
            "inner_pixels": int(ring_masks["inner"].sum()),
            "outer_pixels": int(ring_masks["outer"].sum()),
            "n_psf": len(psfs),
        },
    )

In [5]:

states = [load_galaxy(galaxy) for galaxy in galaxies["galaxy"]]
galaxies["_state"] = states

input_summary_df = pd.DataFrame([state.load_row for state in states])
input_files_df = pd.concat(
    [state.input_manifest for state in states], ignore_index=True
)

display(input_summary_df)
display(input_files_df[["galaxy", "name", "file"]])


,galaxy,ready,inner_pixels,outer_pixels,n_psf
0,NGC 3379,True,646663,1912953,5
1,NGC 1380,True,646765,1940060,5
2,NGC 4486,True,640239,2070508,5


,galaxy,name,file
0,NGC 3379,signal,jw03055-o013_t013_nircam_clear-f150w_i2d.fits
1,NGC 3379,model,jw03055-o013_t013_nircam_clear-f150w_i2d_sbf_m...
2,NGC 3379,catalog_mask,jw03055-o013_t013_nircam_clear-f150w_i2d_sbf_c...
3,NGC 3379,production_residual,jw03055-o013_t013_nircam_clear-f150w_i2d_sbf_r...
4,NGC 3379,inner_ring,jw03055-o013_t013_nircam_clear-f150w_i2d_sbf_r...
5,NGC 3379,outer_ring,jw03055-o013_t013_nircam_clear-f150w_i2d_sbf_r...
6,NGC 3379,measurements,jw03055-o013_t013_nircam_clear-f150w_i2d_sbf2_...
7,NGC 3379,psf_ensemble,jw03055-o013_t013_nircam_clear-f150w_i2d_psf_1...
8,NGC 1380,signal,jw03055-o001_t001_nircam_clear-f150w_i2d.fits
9,NGC 1380,model,jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_m...


## 4. Пороги винзорирования и фоновые области

Сначала для всех трёх объектов восстанавливаются реальные границы
3.0σ, 3.5σ и 4.0σ, проверяется совпадение принятой ветви 3.5σ с
production-FITS и выбираются участки для оценки $N(k)$. Таблица
показывает только диагностически полезные столбцы. Координаты, параметры
фоновой плоскости и служебные WHT-поля остаются в CSV.

In [6]:
def prepare_noise_and_winsor(s):
    GALAXY = s.galaxy
    galaxy_tag = s.galaxy_tag
    table_dir, figure_dir = s.table_dir, s.figure_dir
    signal_path = s.signal_path
    model, weight_map = s.model, s.weight_map
    saved_residual, residual_raw = s.saved_residual, s.residual_raw
    science_mask, ring_masks = s.science_mask, s.ring_masks

    saved_finite = np.isfinite(saved_residual)
    finite_mask_mismatch = int(np.count_nonzero(science_mask ^ saved_finite))

    science_values = residual_raw[science_mask]
    winsor_rows = []
    winsor_limits = {}

    for variant, sigma_limit in WINSOR_VARIANTS:
        if sigma_limit is None:
            median = float(np.median(science_values))
            scale = float(np.std(science_values))
            lower, upper = -np.inf, np.inf
            changed = 0
        else:
            _, median, scale = sigma_clipped_stats(
                science_values, sigma=sigma_limit, maxiters=5
            )
            median = float(median)
            scale = float(scale)
            lower = median - sigma_limit * scale
            upper = median + sigma_limit * scale
            changed = int(np.count_nonzero(
                (science_values < lower) | (science_values > upper)
            ))

        winsor_limits[variant] = (lower, upper)
        winsor_rows.append({
            "variant": variant,
            "sigma_limit": sigma_limit,
            "median": median,
            "scale": scale,
            "lower": lower,
            "upper": upper,
            "changed_pixels": changed,
            "changed_fraction": changed / science_values.size,
        })

    winsor_thresholds = pd.DataFrame(winsor_rows)
    winsor_thresholds.insert(0, "galaxy", GALAXY)
    winsor_thresholds.to_csv(
        table_dir / f"{galaxy_tag}_winsor_thresholds.csv", index=False
    )

    comparison_mask = science_mask & saved_finite
    lower_35, upper_35 = winsor_limits["winsor_3.5"]
    saved_difference = (
        np.clip(residual_raw[comparison_mask], lower_35, upper_35)
        - saved_residual[comparison_mask]
    )
    maximum_difference = float(np.max(np.abs(saved_difference)))

    reconstruction_check = pd.DataFrame([{
        "galaxy": GALAXY,
        "finite_mask_mismatch_pixels": finite_mask_mismatch,
        "compared_pixels": int(comparison_mask.sum()),
        "median_absolute_difference": float(
            np.median(np.abs(saved_difference))
        ),
        "maximum_absolute_difference": maximum_difference,
        "passed": (
            finite_mask_mismatch == 0
            and maximum_difference <= REPRO_PIXEL_TOL
        ),
    }])
    reconstruction_check.to_csv(
        table_dir / f"{galaxy_tag}_winsor_reconstruction_check.csv",
        index=False,
    )

    if not reconstruction_check["passed"].all():
        raise RuntimeError("Ветка 3.5 sigma не совпала с сохранённым FITS")

    ny_full, nx_full = residual_raw.shape
    noise_edges = np.linspace(0.0, np.sqrt(0.5), K_BINS)
    noise_k = 0.5 * (noise_edges[:-1] + noise_edges[1:])

    reference_wht = float(np.median(weight_map[ring_masks["outer"]]))
    tile_rows = []
    candidate_pool = []

    with fits.open(signal_path, memmap=True) as hdul:
        signal_data = hdul["SCI"].data
        signal_error = hdul["ERR"].data
        signal_wht = hdul["WHT"].data

        y_starts = range(
            TILE_MARGIN,
            ny_full - TILE_MARGIN - TILE_SIZE + 1,
            TILE_SIZE,
        )
        x_starts = range(
            TILE_MARGIN,
            nx_full - TILE_MARGIN - TILE_SIZE + 1,
            TILE_SIZE,
        )

        for y0 in y_starts:
            for x0 in x_starts:
                tile_name = f"y{y0}_x{x0}"
                patch = np.array(
                    signal_data[y0:y0 + TILE_SIZE, x0:x0 + TILE_SIZE],
                    dtype=float,
                )
                patch_wht = np.array(
                    signal_wht[y0:y0 + TILE_SIZE, x0:x0 + TILE_SIZE],
                    dtype=float,
                )
                patch_error = np.array(
                    signal_error[y0:y0 + TILE_SIZE, x0:x0 + TILE_SIZE],
                    dtype=float,
                )
                patch_model = np.asarray(
                    model[y0:y0 + TILE_SIZE, x0:x0 + TILE_SIZE],
                    dtype=float,
                )
                yy, xx = np.indices(patch.shape, dtype=float)

                valid = (
                    np.isfinite(patch)
                    & np.isfinite(patch_error)
                    & (patch_error > 0)
                    & np.isfinite(patch_wht)
                    & (patch_wht > 0)
                    & (~np.isfinite(patch_model) | (patch_model <= 0))
                )
                if int(valid.sum()) < 1000:
                    continue

                design = np.column_stack([
                    np.ones(int(valid.sum())),
                    xx[valid],
                    yy[valid],
                ])
                plane_coeff = np.linalg.lstsq(
                    design, patch[valid], rcond=None
                )[0]
                plane = (
                    plane_coeff[0]
                    + plane_coeff[1] * xx
                    + plane_coeff[2] * yy
                )
                standardized = (patch - plane) / patch_error

                median = float(np.median(standardized[valid]))
                source_pixels = valid & (
                    np.abs(standardized - median) > 5.0
                )
                source_pixels = ndimage.binary_dilation(
                    source_pixels, iterations=3
                )
                valid &= ~source_pixels
                if int(valid.sum()) < 1000:
                    continue

                design = np.column_stack([
                    np.ones(int(valid.sum())),
                    xx[valid],
                    yy[valid],
                ])
                plane_coeff = np.linalg.lstsq(
                    design, patch[valid], rcond=None
                )[0]
                plane = (
                    plane_coeff[0]
                    + plane_coeff[1] * xx
                    + plane_coeff[2] * yy
                )
                standardized = (patch - plane) / patch_error

                valid_fraction = float(valid.mean())
                median_wht = float(np.median(patch_wht[valid]))
                wht_ratio = median_wht / reference_wht
                wht_relative_mad = float(
                    1.4826
                    * np.median(np.abs(patch_wht[valid] - median_wht))
                    / median_wht
                )
                standardized_sigma = float(
                    1.4826 * np.median(
                        np.abs(
                            standardized[valid]
                            - np.median(standardized[valid])
                        )
                    )
                )
                wht_ok = (
                    0.5 <= wht_ratio <= 2.0
                    and wht_relative_mad <= 0.25
                )
                accepted = (
                    valid_fraction >= MIN_TILE_VALID_FRACTION and wht_ok
                )
                relaxed = (
                    valid_fraction >= RELAXED_TILE_VALID_FRACTION and wht_ok
                )

                row = {
                    "tile": tile_name,
                    "y0": y0,
                    "x0": x0,
                    "size": TILE_SIZE,
                    "valid_fraction": valid_fraction,
                    "median_wht": median_wht,
                    "wht_ratio_to_outer_ring": wht_ratio,
                    "wht_relative_mad": wht_relative_mad,
                    "median_ERR": float(np.median(patch_error[valid])),
                    "standardized_sigma": standardized_sigma,
                    "plane_0": float(plane_coeff[0]),
                    "plane_x": float(plane_coeff[1]),
                    "plane_y": float(plane_coeff[2]),
                    "accepted": accepted,
                    "selected": False,
                }
                tile_rows.append(row)
                if not relaxed:
                    continue

                hann = np.outer(
                    np.hanning(TILE_SIZE),
                    np.hanning(TILE_SIZE),
                )
                window = hann * valid
                clean = np.zeros_like(standardized)
                clean[valid] = standardized[valid]
                clean -= np.sum(clean * window) / np.sum(window)
                power = np.abs(fft2(clean * window))**2 / np.sum(window**2)

                ky = fftfreq(TILE_SIZE)[:, None]
                kx = fftfreq(TILE_SIZE)[None, :]
                radius = np.hypot(kx, ky)
                profile = np.full(noise_k.size, np.nan)
                for index, (low, high) in enumerate(
                    zip(noise_edges[:-1], noise_edges[1:])
                ):
                    selected = (
                        (radius >= low)
                        & (radius < high)
                        & np.isfinite(power)
                    )
                    if int(selected.sum()) >= 10:
                        profile[index] = float(np.mean(power[selected]))

                fit_range = (
                    (noise_k >= K_MIN)
                    & (noise_k <= K_MAX)
                    & np.isfinite(profile)
                    & (profile > 0)
                )
                profile /= np.median(profile[fit_range])
                candidate_pool.append({
                    "tile": tile_name,
                    "y0": y0,
                    "x0": x0,
                    "valid_fraction": valid_fraction,
                    "nominal": accepted,
                    "profile": profile,
                    "standardized": standardized,
                    "valid": valid,
                })

    nominal_candidates = [item for item in candidate_pool if item["nominal"]]
    if len(nominal_candidates) >= 3:
        remaining = nominal_candidates.copy()
        noise_tile_mode = "nominal_0.80"
    else:
        remaining = candidate_pool.copy()
        noise_tile_mode = "relaxed_0.55"
        print(
            "ВНИМАНИЕ: чистых областей меньше трёх; "
            "N(k) строится в диагностическом режиме >= 0.55"
        )
    selected_candidates = []
    if remaining:
        first = max(remaining, key=lambda item: item["valid_fraction"])
        selected_candidates.append(first)
        remaining.remove(first)

    while remaining and len(selected_candidates) < MAX_NOISE_TILES:
        next_tile = max(
            remaining,
            key=lambda item: min(
                (item["y0"] - chosen["y0"])**2
                + (item["x0"] - chosen["x0"])**2
                for chosen in selected_candidates
            ),
        )
        selected_candidates.append(next_tile)
        remaining.remove(next_tile)

    selected_names = {item["tile"] for item in selected_candidates}
    for row in tile_rows:
        row["selected"] = row["tile"] in selected_names

    selected_profiles = {
        item["tile"]: item["profile"]
        for item in selected_candidates
    }
    tile_views = [
        (item["tile"], item["standardized"], item["valid"])
        for item in selected_candidates
    ]
    tile_mask_hdus = [fits.PrimaryHDU()]
    for index, item in enumerate(selected_candidates, start=1):
        tile_mask_hdus.append(
            fits.CompImageHDU(
                data=item["valid"].astype(np.uint8),
                name=f"TILE{index:02d}",
            )
        )
        saved_noise = np.full(
            item["standardized"].shape,
            np.nan,
            dtype=np.float32,
        )
        saved_noise[item["valid"]] = item["standardized"][item["valid"]]
        tile_mask_hdus.append(
            fits.CompImageHDU(data=saved_noise, name=f"NOISE{index:02d}")
        )

    noise_tiles = pd.DataFrame(tile_rows)
    noise_tiles.insert(0, "galaxy", GALAXY)
    noise_tiles["selection_mode"] = noise_tile_mode
    if not selected_profiles:
        raise RuntimeError("На кадре не найдено ни одной пригодной шумовой области")

    profile_stack = np.vstack(list(selected_profiles.values()))
    noise_median = np.nanmedian(profile_stack, axis=0)
    noise_p16 = np.nanpercentile(profile_stack, 16, axis=0)
    noise_p84 = np.nanpercentile(profile_stack, 84, axis=0)
    noise_fit = (
        (noise_k >= K_MIN) & (noise_k <= K_MAX)
        & np.isfinite(noise_median) & (noise_median > 0)
    )
    noise_normalization = np.median(noise_median[noise_fit])
    noise_median /= noise_normalization
    noise_p16 /= noise_normalization
    noise_p84 /= noise_normalization

    noise_profile = pd.DataFrame({
        "galaxy": GALAXY,
        "k": noise_k,
        "N_reference": noise_median,
        "N_p16": noise_p16,
        "N_p84": noise_p84,
    })
    for tile_name, profile in selected_profiles.items():
        noise_profile[tile_name] = profile

    noise_tiles.to_csv(
        table_dir / f"{galaxy_tag}_noise_tiles.csv", index=False
    )
    noise_profile.to_csv(
        table_dir / f"{galaxy_tag}_noise_profiles.csv", index=False
    )
    fits.HDUList(tile_mask_hdus).writeto(
        table_dir / f"{galaxy_tag}_noise_tiles.fits",
        overwrite=True,
    )

    s.winsor_limits = winsor_limits
    s.winsor_thresholds = winsor_thresholds
    s.reconstruction_check = reconstruction_check
    s.noise_tiles = noise_tiles
    s.noise_profile = noise_profile
    s.noise_tile_mode = noise_tile_mode
    s.tile_views = tile_views

In [7]:

for state in states:
    prepare_noise_and_winsor(state)

winsor_thresholds_df = pd.concat(
    [state.winsor_thresholds for state in states], ignore_index=True
)
reproduction_df = pd.concat(
    [state.reconstruction_check for state in states], ignore_index=True
)
noise_tiles_df = pd.concat(
    [state.noise_tiles for state in states], ignore_index=True
)
noise_profiles_df = pd.concat(
    [state.noise_profile[["galaxy", "k", "N_reference", "N_p16", "N_p84"]]
     for state in states],
    ignore_index=True,
)
galaxies["noise_tile_mode"] = [state.noise_tile_mode for state in states]

threshold_view = winsor_thresholds_df.dropna(subset=["sigma_limit"])[
    ["galaxy", "sigma_limit", "lower", "upper", "changed_fraction"]
].copy()
threshold_view["changed_fraction"] *= 100
threshold_view = threshold_view.rename(columns={
    "sigma_limit": "sigma",
    "lower": "lower_MJy_sr",
    "upper": "upper_MJy_sr",
    "changed_fraction": "changed_percent",
})

selected_tiles_view = noise_tiles_df[noise_tiles_df["selected"]][[
    "galaxy", "tile", "valid_fraction", "wht_ratio_to_outer_ring",
    "standardized_sigma", "selection_mode",
]]
display(threshold_view)
display(reproduction_df[[
    "galaxy", "finite_mask_mismatch_pixels",
    "maximum_absolute_difference", "passed",
]])
display(selected_tiles_view)


ВНИМАНИЕ: чистых областей меньше трёх; N(k) строится в диагностическом режиме >= 0.55


,galaxy,sigma,lower_MJy_sr,upper_MJy_sr,changed_percent
1,NGC 3379,3.0,-3.330169,2.818035,4.635940
2,NGC 3379,3.5,-4.203014,3.707736,2.552355
3,NGC 3379,4.0,-5.077862,4.591052,1.504703
5,NGC 1380,3.0,-1.719784,1.570354,3.524386
6,NGC 1380,3.5,-2.138215,1.995796,1.932363
7,NGC 1380,4.0,-2.544443,2.405369,1.201753
9,NGC 4486,3.0,-2.757587,2.518125,2.717132
10,NGC 4486,3.5,-3.419561,3.184525,1.225758
11,NGC 4486,4.0,-4.048743,3.816489,0.585245


,galaxy,finite_mask_mismatch_pixels,maximum_absolute_difference,passed
0,NGC 3379,0,0.000000e+00,True
1,NGC 1380,0,1.192093e-07,True
2,NGC 4486,0,0.000000e+00,True


,galaxy,tile,valid_fraction,wht_ratio_to_outer_ring,standardized_sigma,selection_mode
0,NGC 3379,y96_x96,0.854645,0.960861,0.792909,nominal_0.80
35,NGC 3379,y1120_x608,0.812931,0.977473,0.800654,nominal_0.80
72,NGC 3379,y2656_x96,0.850105,0.997824,0.798672,nominal_0.80
97,NGC 3379,y3680_x608,0.815151,1.007902,0.815189,nominal_0.80
110,NGC 1380,y96_x1120,0.803474,0.998690,0.827509,nominal_0.80
130,NGC 1380,y1120_x96,0.872280,1.016668,0.783935,nominal_0.80
166,NGC 1380,y2656_x1120,0.830128,1.036868,0.806769,nominal_0.80
176,NGC 1380,y3168_x96,0.893944,1.043374,0.791220,nominal_0.80
259,NGC 4486,y2656_x8288,0.560184,1.027255,1.071203,relaxed_0.55
260,NGC 4486,y2656_x8800,0.576939,1.027480,1.034915,relaxed_0.55


### Подготовка распределений яркости

Для каждого кольца исходные остатки раскладываются по бинам. В
`pixel_histogram_df` одна строка соответствует одному бину и содержит его
границы, центр, абсолютное число пикселей и цветовую зону винзорирования.

Пороги остаются глобальными — ровно теми, которые действительно применяет
pipeline. Они не пересчитываются отдельно для колец, поэтому этот блок не
меняет production-методику.

In [8]:
zone_colors = {
    "blue": "#2563eb",
    "red": "#dc2626",
    "yellow": "#facc15",
    "orange": "#f97316",
}

threshold_rows = []
band_rows = []
histogram_rows = []
histogram_summary_rows = []

for state in states:
    limits = state.winsor_limits
    lower3, upper3 = limits["winsor_3.0"]
    lower35, upper35 = limits["winsor_3.5"]
    lower4, upper4 = limits["winsor_4.0"]

    science_values = state.residual_raw[state.science_mask]
    for variant, sigma in [
        ("winsor_3.0", 3.0),
        ("winsor_3.5", 3.5),
        ("winsor_4.0", 4.0),
    ]:
        lower, upper = limits[variant]
        below = 100 * np.mean(science_values < lower)
        above = 100 * np.mean(science_values > upper)
        threshold_rows.append({
            "galaxy": state.galaxy,
            "sigma": sigma,
            "lower_MJy_sr": lower,
            "upper_MJy_sr": upper,
            "below_percent": below,
            "above_percent": above,
            "total_percent": below + above,
        })

    center = 0.5 * (lower35 + upper35)
    scale = (upper35 - lower35) / 7.0
    shown_min = center - 6 * scale
    shown_max = center + 6 * scale

    for ring, mask in state.ring_masks.items():
        values = state.residual_raw[mask]
        shown = np.clip(values, shown_min, shown_max)
        regular_edges = np.linspace(shown_min, shown_max, 181)
        sigma_edges = np.array([
            lower4, lower35, lower3, upper3, upper35, upper4
        ])
        sigma_edges = sigma_edges[
            (sigma_edges > shown_min) & (sigma_edges < shown_max)
        ]
        edges = np.unique(np.r_[regular_edges, sigma_edges])
        counts, edges = np.histogram(shown, bins=edges)
        centers = 0.5 * (edges[:-1] + edges[1:])

        zones = np.full(centers.size, "orange", dtype=object)
        zones[(centers >= lower4) & (centers <= upper4)] = "yellow"
        zones[(centers >= lower35) & (centers <= upper35)] = "red"
        zones[(centers >= lower3) & (centers <= upper3)] = "blue"

        for left, right, bin_center, count, zone in zip(
            edges[:-1], edges[1:], centers, counts, zones
        ):
            histogram_rows.append({
                "galaxy": state.galaxy,
                "ring": ring,
                "brightness_left_MJy_sr": left,
                "brightness_right_MJy_sr": right,
                "brightness_center_MJy_sr": bin_center,
                "pixel_count": int(count),
                "zone": zone,
            })

        outside3 = np.mean((values < lower3) | (values > upper3))
        outside35 = np.mean((values < lower35) | (values > upper35))
        outside4 = np.mean((values < lower4) | (values > upper4))
        band_rows.append({
            "galaxy": state.galaxy,
            "ring": ring,
            "blue_percent": 100 * (1 - outside3),
            "red_percent": 100 * max(outside3 - outside35, 0),
            "yellow_percent": 100 * max(outside35 - outside4, 0),
            "orange_percent": 100 * outside4,
        })
        histogram_summary_rows.append({
            "galaxy": state.galaxy,
            "ring": ring,
            "n_pixels": len(values),
            "shown_min_MJy_sr": shown_min,
            "shown_max_MJy_sr": shown_max,
            "left_overflow_percent": 100 * np.mean(values < shown_min),
            "right_overflow_percent": 100 * np.mean(values > shown_max),
        })

winsor_tail_df = pd.DataFrame(threshold_rows)
winsor_band_df = pd.DataFrame(band_rows)
pixel_histogram_df = pd.DataFrame(histogram_rows)
histogram_summary_df = pd.DataFrame(histogram_summary_rows)

winsor_tail_df.to_csv(
    pilot_table_dir / "sbf2_winsor_tail_fractions.csv", index=False
)
winsor_band_df.to_csv(
    pilot_table_dir / "sbf2_winsor_color_bands.csv", index=False
)
pixel_histogram_df.to_csv(
    pilot_table_dir / "sbf2_pixel_brightness_histograms.csv", index=False
)

display(winsor_tail_df)
display(winsor_band_df)
display(histogram_summary_df)

,galaxy,sigma,lower_MJy_sr,upper_MJy_sr,below_percent,above_percent,total_percent
0,NGC 3379,3.0,-3.330169,2.818035,1.505363,3.130577,4.635940
1,NGC 3379,3.5,-4.203014,3.707736,0.830159,1.722196,2.552355
2,NGC 3379,4.0,-5.077862,4.591052,0.487587,1.017116,1.504703
3,NGC 1380,3.0,-1.719784,1.570354,1.224654,2.299732,3.524386
4,NGC 1380,3.5,-2.138215,1.995796,0.717472,1.214891,1.932363
5,NGC 1380,4.0,-2.544443,2.405369,0.488530,0.713224,1.201753
6,NGC 4486,3.0,-2.757587,2.518125,1.064574,1.652558,2.717132
7,NGC 4486,3.5,-3.419561,3.184525,0.425925,0.799833,1.225758
8,NGC 4486,4.0,-4.048743,3.816489,0.172588,0.412657,0.585245


,galaxy,ring,blue_percent,red_percent,yellow_percent,orange_percent
0,NGC 3379,inner,90.121593,5.564104,2.441921,1.872382
1,NGC 3379,outer,99.197053,0.618154,0.144436,0.040356
2,NGC 1380,inner,94.201294,3.524773,1.365411,0.908522
3,NGC 1380,outer,99.256157,0.545447,0.144016,0.054380
4,NGC 4486,inner,86.700123,7.335854,3.382955,2.581067
5,NGC 4486,outer,98.726255,1.051046,0.183482,0.039217


,galaxy,ring,n_pixels,shown_min_MJy_sr,shown_max_MJy_sr,left_overflow_percent,right_overflow_percent
0,NGC 3379,inner,646663,-7.028282,6.533004,0.002165,0.294435
1,NGC 3379,outer,1912953,-7.028282,6.533004,0.000000,0.001307
2,NGC 1380,inner,646765,-3.614648,3.472229,0.000773,0.082719
3,NGC 1380,outer,1940060,-3.614648,3.472229,0.000155,0.001907
4,NGC 4486,inner,640239,-5.778164,5.543127,0.017181,0.323785
5,NGC 4486,outer,2070508,-5.778164,5.543127,0.000000,0.000193


### Отдельные гистограммы яркости пикселей

Следующая ячейка создаёт **шесть самостоятельных рисунков**: отдельный
`Figure` для каждой галактики и каждого кольца.

- ось X — остаточная яркость пикселя в MJy sr$^{-1}$;
- ось Y — абсолютное количество пикселей в бине, без логарифма;
- синий — сохраняется при 3σ;
- красный — режется при 3σ, но сохраняется при 3.5σ;
- жёлтый — режется при 3.5σ, но сохраняется при 4σ;
- оранжевый — режется даже при 4σ.

Чтобы единичные экстремальные выбросы не сжали центральную часть графика,
значения за показанным диапазоном собраны в двух крайних бинах. Их доля
указана в `histogram_summary_df`; общее число пикселей при этом сохраняется.

In [9]:
ring_names = {"inner": "внутреннее кольцо", "outer": "внешнее кольцо"}
legend_handles = [
    Patch(color=zone_colors["blue"], label="сохраняется при 3σ"),
    Patch(color=zone_colors["red"], label="режется при 3σ"),
    Patch(color=zone_colors["yellow"], label="режется при 3.5σ"),
    Patch(color=zone_colors["orange"], label="режется при 4σ"),
]

for state in states:
    for ring in ["inner", "outer"]:
        histogram = pixel_histogram_df[
            pixel_histogram_df["galaxy"].eq(state.galaxy)
            & pixel_histogram_df["ring"].eq(ring)
        ]

        fig, axis = plt.subplots(figsize=(9, 5.5))
        axis.bar(
            histogram["brightness_center_MJy_sr"],
            histogram["pixel_count"],
            width=(
                histogram["brightness_right_MJy_sr"]
                - histogram["brightness_left_MJy_sr"]
            ),
            color=histogram["zone"].map(zone_colors),
            linewidth=0,
        )
        axis.set(
            title=f"{state.galaxy}: {ring_names[ring]}",
            xlabel=r"Остаточная яркость пикселя (MJy sr$^{-1}$)",
            ylabel="Количество пикселей",
        )
        axis.set_ylim(bottom=0)
        axis.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
        axis.legend(handles=legend_handles, frameon=False)
        fig.tight_layout()
        fig.savefig(
            state.figure_dir
            / f"{state.galaxy_tag}_{ring}_pixel_brightness_histogram.pdf",
            bbox_inches="tight",
        )
        plt.show()
        plt.close(fig)

# Те же распределения для всей области, по которой заданы глобальные пороги.
full_histogram_rows = []

for state in states:
    values = state.residual_raw[state.science_mask]
    lower3, upper3 = state.winsor_limits["winsor_3.0"]
    lower35, upper35 = state.winsor_limits["winsor_3.5"]
    lower4, upper4 = state.winsor_limits["winsor_4.0"]

    center = 0.5 * (lower35 + upper35)
    scale = (upper35 - lower35) / 7.0
    shown = np.clip(values, center - 6 * scale, center + 6 * scale)
    counts, edges = np.histogram(shown, bins=180)
    centers = 0.5 * (edges[:-1] + edges[1:])

    zones = np.full(centers.size, "orange", dtype=object)
    zones[(centers >= lower4) & (centers <= upper4)] = "yellow"
    zones[(centers >= lower35) & (centers <= upper35)] = "red"
    zones[(centers >= lower3) & (centers <= upper3)] = "blue"

    for left, right, bin_center, count, zone in zip(
        edges[:-1], edges[1:], centers, counts, zones
    ):
        full_histogram_rows.append({
            "galaxy": state.galaxy,
            "brightness_left_MJy_sr": left,
            "brightness_right_MJy_sr": right,
            "brightness_center_MJy_sr": bin_center,
            "pixel_count": int(count),
            "zone": zone,
        })

full_sbf_histogram_df = pd.DataFrame(full_histogram_rows)

for state in states:
    histogram = full_sbf_histogram_df[
        full_sbf_histogram_df["galaxy"].eq(state.galaxy)
    ]

    fig, axis = plt.subplots(figsize=(9, 5.5))
    axis.bar(
        histogram["brightness_center_MJy_sr"],
        histogram["pixel_count"],
        width=(
            histogram["brightness_right_MJy_sr"]
            - histogram["brightness_left_MJy_sr"]
        ),
        color=histogram["zone"].map(zone_colors),
        linewidth=0,
    )
    axis.set(
        title=f"{state.galaxy}: вся SBF-область",
        xlabel=r"Остаточная яркость пикселя (MJy sr$^{-1}$)",
        ylabel="Количество пикселей",
    )
    axis.set_ylim(bottom=0)
    axis.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    axis.legend(handles=legend_handles, frameon=False)
    fig.tight_layout()
    fig.savefig(
        state.figure_dir
        / f"{state.galaxy_tag}_full_sbf_pixel_brightness_histogram.pdf",
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)


/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/3942010555.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/3942010555.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/3942010555.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/3942010555.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/3942010555.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/3942010555.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/3942010555.py:108: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/3942010555.py:108: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/3942010555.py:108: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Pixel-brightness distributions (English figure set)

Each galaxy is shown in three separate figures: the full SBF region, the
inner annulus, and the outer annulus. The horizontal axis is the residual
pixel brightness and the vertical axis is the absolute number of pixels.
The colours show which tails are affected by the tested winsorization
thresholds. At the adopted $3.5\sigma$ threshold the yellow and orange
fractions are capped, not removed; their percentage is printed in each
panel. These figures use the same bins, masks, and global thresholds as the
Russian diagnostic figures above.


In [10]:
english_zone_legend = [
    Patch(color=zone_colors["blue"], label=r"retained at $3\sigma$"),
    Patch(color=zone_colors["red"], label=r"capped at $3\sigma$"),
    Patch(color=zone_colors["yellow"], label=r"capped at $3.5\sigma$"),
    Patch(color=zone_colors["orange"], label=r"capped at $4\sigma$"),
]

for state in states:
    full_fraction = winsor_tail_df.loc[
        winsor_tail_df["galaxy"].eq(state.galaxy)
        & winsor_tail_df["sigma"].eq(3.5),
        "total_percent",
    ].iloc[0]

    regions = [
        (
            "full_sbf",
            "Full SBF region",
            full_sbf_histogram_df[
                full_sbf_histogram_df["galaxy"].eq(state.galaxy)
            ],
            full_fraction,
        ),
    ]
    for ring, title in [("inner", "Inner annulus"),
                        ("outer", "Outer annulus")]:
        histogram = pixel_histogram_df[
            pixel_histogram_df["galaxy"].eq(state.galaxy)
            & pixel_histogram_df["ring"].eq(ring)
        ]
        bands = winsor_band_df[
            winsor_band_df["galaxy"].eq(state.galaxy)
            & winsor_band_df["ring"].eq(ring)
        ].iloc[0]
        affected = bands["yellow_percent"] + bands["orange_percent"]
        regions.append((ring, title, histogram, affected))

    for region, title, histogram, affected in regions:
        fig, axis = plt.subplots(figsize=(9, 5.5))
        axis.bar(
            histogram["brightness_center_MJy_sr"],
            histogram["pixel_count"],
            width=(
                histogram["brightness_right_MJy_sr"]
                - histogram["brightness_left_MJy_sr"]
            ),
            color=histogram["zone"].map(zone_colors),
            linewidth=0,
        )
        axis.set(
            title=f"{state.galaxy}: {title}",
            xlabel=r"Residual pixel brightness (MJy sr$^{-1}$)",
            ylabel="Number of pixels",
        )
        axis.set_ylim(bottom=0)
        axis.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
        axis.text(
            0.98, 0.96,
            rf"Pixels capped at $3.5\sigma$: {affected:.2f}%",
            transform=axis.transAxes, ha="right", va="top",
        )
        axis.legend(handles=english_zone_legend, frameon=False)
        fig.tight_layout()
        fig.savefig(
            state.figure_dir
            / f"{state.galaxy_tag}_{region}_pixel_brightness_histogram_english.pdf",
            bbox_inches="tight",
        )
        plt.show()
        plt.close(fig)


/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/1857629012.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/1857629012.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/1857629012.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/1857629012.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/1857629012.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/1857629012.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/1857629012.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/1857629012.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/1857629012.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Как читать фоновые области и $N(k)$

Первый рисунок показывает четыре участка на галактику, используемые для
оценки шума. Белые провалы — исключённые или NaN-пиксели. Остаточные
объекты и крупномасштабная структура означают, что участок нельзя считать
чистым детекторным шумом. Для NGC 4486 режим `relaxed_0.55` прямо указан
в таблице.

На втором рисунке линия — медианный спектр выбранных участков, полоса —
диапазон 16–84%. Горизонталь $N=1$ соответствует белому шуму. Избыток
мощности сам по себе ещё не означает, что $N(k)$ можно независимо
отделить от $E(k)$.

In [11]:

fig, axes = plt.subplots(3, MAX_NOISE_TILES, figsize=(12, 8))
for row, state in enumerate(states):
    for axis in axes[row]:
        axis.set_axis_off()
    for axis, (tile_name, standardized, valid) in zip(axes[row], state.tile_views):
        shown = np.where(valid, standardized, np.nan)
        limit = np.nanpercentile(np.abs(shown), 98)
        axis.imshow(shown, origin="lower", cmap="gray", vmin=-limit, vmax=limit)
        axis.set_title(tile_name.replace("_", ", "), fontsize=9)
        axis.set_axis_off()
    axes[row, 0].set_ylabel(
        f"{state.galaxy}\n{state.noise_tile_mode}", rotation=0,
        ha="right", va="center", labelpad=35,
    )
fig.suptitle("Участки для оценки коррелированного шума")
fig.tight_layout(rect=(0, 0, 1, 0.96))
fig.savefig(pilot_figure_dir / "sbf2_noise_tiles_all_galaxies.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for axis, state in zip(axes, states):
    profile = state.noise_profile
    axis.fill_between(profile["k"], profile["N_p16"], profile["N_p84"],
                      color="#93c5fd", alpha=0.45)
    axis.plot(profile["k"], profile["N_reference"], color="#1d4ed8", lw=2)
    axis.axhline(1, color="black", lw=1, ls="--")
    axis.set(xlim=(0.01, 0.30), title=state.galaxy,
             xlabel=r"$k$ (pixel$^{-1}$)")
axes[0].set_ylabel("Нормированная мощность")
fig.tight_layout()
fig.savefig(pilot_figure_dir / "sbf2_noise_profiles_all_galaxies.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)


/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/549192494.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/549192494.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Построение $E(k)$, $N(k)$ и измеренного спектра

Эта ячейка выполняет FFT-этап сначала для всех трёх галактик, затем
собирает результаты в длинные таблицы. На экран выводятся только шесть
строк геометрии — по два кольца на объект. Сами спектры остаются доступны
в `expectation_df` и `power_spectra_df`.

In [12]:
def build_templates_and_spectra(s):
    GALAXY = s.galaxy
    galaxy_tag, table_dir = s.galaxy_tag, s.table_dir
    model, error_map = s.model, s.error_map
    residual_raw, ring_masks = s.residual_raw, s.ring_masks
    psf_ids, psfs = s.psf_ids, s.psfs
    main_measurements, region_name = s.main_measurements, s.region_name
    noise_profile, winsor_limits = s.noise_profile, s.winsor_limits

    prepared_rings = {}

    for ring in ring_masks:
        production_row = main_measurements[
            main_measurements["region"].eq(region_name[ring])
        ].iloc[0]
        y0 = int(production_row["fft_crop_y0"])
        y1 = int(production_row["fft_crop_y1"])
        x0 = int(production_row["fft_crop_x0"])
        x1 = int(production_row["fft_crop_x1"])
        crop = (slice(y0, y1), slice(x0, x1))

        window = ring_masks[ring][crop].copy()
        model_crop = np.asarray(model[crop], dtype=float)
        error_crop = np.asarray(error_map[crop], dtype=float)
        raw_crop = np.asarray(residual_raw[crop], dtype=float)
        window &= (
            np.isfinite(raw_crop)
            & np.isfinite(model_crop)
            & (model_crop > 0)
        )
        valid_error = window & np.isfinite(error_crop) & (error_crop > 0)
        if not valid_error.any():
            raise RuntimeError(f"{ring}: в рабочем кольце нет корректного ERR")
        if valid_error.sum() != window.sum():
            error_fill = float(np.median(error_crop[valid_error]))
            error_crop = np.where(
                np.isfinite(error_crop) & (error_crop > 0),
                error_crop,
                error_fill,
            )
        n_use = int(window.sum())
        mean_model = float(np.mean(model_crop[window]))
        shape = window.shape
        geometry_ok = (
            n_use == int(production_row["n_use"])
            and shape == (
                int(production_row["fft_crop_ny"]),
                int(production_row["fft_crop_nx"]),
            )
            and abs(mean_model / float(production_row["Imean"]) - 1) <= 1e-6
        )
        if not geometry_ok:
            raise RuntimeError(f"{ring}: маска или FFT-crop не совпали")

        ky = fftfreq(shape[0])[:, None]
        kx = fftfreq(shape[1])[None, :]
        radius = np.hypot(kx, ky)
        edges = np.linspace(0.0, float(radius.max()), K_BINS)
        centers = 0.5 * (edges[:-1] + edges[1:])
        ids = np.searchsorted(edges, radius.ravel(), side="right") - 1
        valid_ids = (ids >= 0) & (ids < centers.size)
        radial_plan = {
            "ids": ids,
            "valid": valid_ids,
            "n_bins": centers.size,
        }

        expectation_profiles = []
        for psf_index, psf in enumerate(psfs):
            padded_psf = np.zeros(shape, dtype=float)
            py, px = psf.shape
            psf_y0 = shape[0] // 2 - py // 2
            psf_x0 = shape[1] // 2 - px // 2
            padded_psf[
                psf_y0:psf_y0 + py,
                psf_x0:psf_x0 + px,
            ] = psf

            with set_workers(FFT_WORKERS):
                psf_filter = fft2(padded_psf)
                expectation = monte_carlo_template(
                    window,
                    radial_plan,
                    psf_filter,
                    E_REALIZATIONS,
                    RANDOM_SEED + psf_index,
                )
            expectation_profiles.append(expectation)

        finite_noise = (
            np.isfinite(noise_profile["k"])
            & np.isfinite(noise_profile["N_reference"])
            & (noise_profile["N_reference"] > 0)
        )
        intrinsic_noise = np.interp(
            radius,
            noise_profile.loc[finite_noise, "k"],
            noise_profile.loc[finite_noise, "N_reference"],
        )
        intrinsic_noise = np.maximum(intrinsic_noise, 0)
        noise_filter = np.sqrt(intrinsic_noise)
        noise_amplitude = np.zeros(shape, dtype=float)
        noise_amplitude[window] = (
            error_crop[window] / np.sqrt(model_crop[window])
        )

        with set_workers(FFT_WORKERS):
            noise_windowed = monte_carlo_template(
                window,
                radial_plan,
                noise_filter,
                N_REALIZATIONS,
                RANDOM_SEED + 1000 + (ring == "outer"),
                amplitude=noise_amplitude,
            )

        fit_range = (
            (centers >= K_MIN)
            & (centers <= K_MAX)
            & np.isfinite(noise_windowed)
            & (noise_windowed > 0)
        )
        noise_windowed /= np.median(noise_windowed[fit_range])

        prepared_rings[ring] = {
            "crop": crop,
            "window": window,
            "model": model_crop,
            "raw": raw_crop,
            "n_use": n_use,
            "mean_model": mean_model,
            "shape": shape,
            "k": centers,
            "radial_plan": radial_plan,
            "E_profiles": np.vstack(expectation_profiles),
            "N_profile": noise_windowed,
        }


    expectation_rows = []
    for ring, prepared in prepared_rings.items():
        for psf_id, profile in zip(psf_ids, prepared["E_profiles"]):
            for k_value, e_value, n_value in zip(
                prepared["k"], profile, prepared["N_profile"]
            ):
                expectation_rows.append({
                    "galaxy": GALAXY,
                    "ring": ring,
                    "psf_id": psf_id,
                    "k": k_value,
                    "E": e_value,
                    "N": n_value,
                })
    expectation_spectra = pd.DataFrame(expectation_rows)
    expectation_spectra.to_csv(
        table_dir / f"{galaxy_tag}_expectation_spectra.csv",
        index=False,
    )

    spectrum_rows = []
    variant_rows = []
    spectra = {}

    for ring, prepared in prepared_rings.items():
        crop = prepared["crop"]
        window = prepared["window"]
        model_crop = prepared["model"]

        for variant, _ in WINSOR_VARIANTS:
            residual_crop = np.asarray(residual_raw[crop], dtype=float)
            lower, upper = winsor_limits[variant]
            if np.isfinite(lower) and np.isfinite(upper):
                residual_crop = residual_crop.copy()
                residual_crop[window] = np.clip(
                    residual_crop[window],
                    lower,
                    upper,
                )
                changed = int(np.count_nonzero(
                    window & ((prepared["raw"] < lower) | (prepared["raw"] > upper))
                ))
            else:
                changed = 0
            variant_rows.append({
                "galaxy": GALAXY,
                "ring": ring,
                "variant": variant,
                "changed_pixels": changed,
                "changed_fraction": changed / prepared["n_use"],
            })

            normalized = np.zeros(prepared["shape"], dtype=float)
            normalized[window] = (
                residual_crop[window] / np.sqrt(model_crop[window])
            )
            normalized[window] -= np.mean(normalized[window])

            with set_workers(FFT_WORKERS):
                power = np.abs(fft2(normalized))**2 / prepared["n_use"]

            Pk, Pk_error, Pk_count = radial_mean_sem(
                power,
                prepared["radial_plan"],
            )
            E_median = np.nanmedian(prepared["E_profiles"], axis=0)
            E_mad = (
                1.4826
                * np.nanmedian(
                    np.abs(prepared["E_profiles"] - E_median),
                    axis=0,
                )
            )

            spectra[(ring, variant)] = {
                "Pk": Pk,
                "Pk_error": Pk_error,
                "Pk_count": Pk_count,
            }

            for index, k_value in enumerate(prepared["k"]):
                spectrum_rows.append({
                    "galaxy": GALAXY,
                    "ring": ring,
                    "variant": variant,
                    "k": float(k_value),
                    "Pk": float(Pk[index]),
                    "Pk_error": float(Pk_error[index]),
                    "Pk_count": int(Pk_count[index]),
                    "E_median": float(E_median[index]),
                    "E_mad": float(E_mad[index]),
                    "N_windowed": float(prepared["N_profile"][index]),
                })

    power_spectra = pd.DataFrame(spectrum_rows)
    winsor_by_ring = pd.DataFrame(variant_rows)
    power_spectra.to_csv(
        table_dir / f"{galaxy_tag}_noise_winsor_power_spectra.csv",
        index=False,
    )
    winsor_by_ring.to_csv(
        table_dir / f"{galaxy_tag}_winsor_by_ring.csv", index=False
    )
    s.prepared_rings = prepared_rings
    s.spectra = spectra
    s.expectation_spectra = expectation_spectra
    s.power_spectra = power_spectra
    s.winsor_by_ring = winsor_by_ring
    s.rings = tuple(ring_masks)


In [13]:

for state in states:
    build_templates_and_spectra(state)

expectation_df = pd.concat(
    [state.expectation_spectra for state in states], ignore_index=True
)
power_spectra_df = pd.concat(
    [state.power_spectra for state in states], ignore_index=True
)
winsor_by_ring_df = pd.concat(
    [state.winsor_by_ring for state in states], ignore_index=True
)

spectrum_summary_df = pd.DataFrame([
    {
        "galaxy": state.galaxy,
        "ring": ring,
        "crop": str(prepared["shape"]),
        "n_pixels": prepared["n_use"],
        "n_psf": len(state.psf_ids),
        "n_k_bins": len(prepared["k"]),
    }
    for state in states
    for ring, prepared in state.prepared_rings.items()
])
display(spectrum_summary_df)


,galaxy,ring,crop,n_pixels,n_psf,n_k_bins
0,NGC 3379,inner,"(1309, 1308)",646663,5,79
1,NGC 3379,outer,"(2359, 2359)",1912953,5,79
2,NGC 1380,inner,"(1309, 1308)",646765,5,79
3,NGC 1380,outer,"(2349, 2358)",1940060,5,79
4,NGC 4486,inner,"(1308, 1308)",640239,5,79
5,NGC 4486,outer,"(2359, 2359)",2070508,5,79


### Сравнение форм $E(k)$ и $N(k)$

Обе кривые нормированы на медиану внутри рабочего диапазона, поэтому
рисунок сравнивает именно **форму**, а не амплитуду. Чем ближе красная и
синяя линии и чем ближе корреляция к единице, тем хуже двухпараметрический
фит способен разделить SBF и коррелированный шум. Это диагностика
вырождения шаблонов, а не самостоятельное измерение SBF.

In [14]:

fig, axes = plt.subplots(3, 2, figsize=(11, 10), sharex=True)
template_rows = []
for row, state in enumerate(states):
    for column, ring in enumerate(["inner", "outer"]):
        prepared = state.prepared_rings[ring]
        k = prepared["k"]
        E = np.nanmedian(prepared["E_profiles"], axis=0)
        N = prepared["N_profile"]
        selected = ((k >= K_MIN) & (k <= K_MAX)
                    & np.isfinite(E) & np.isfinite(N))
        E = E / np.median(E[selected])
        N = N / np.median(N[selected])
        correlation = np.corrcoef(E[selected], N[selected])[0, 1]
        template_rows.append({
            "galaxy": state.galaxy, "ring": ring,
            "E_N_correlation": correlation,
        })
        axis = axes[row, column]
        axis.plot(k, E, color="#dc2626", lw=2, label=r"$E(k)$")
        axis.plot(k, N, color="#2563eb", lw=2, label=r"$N(k)$")
        axis.axvspan(K_MIN, K_MAX, color="0.8", alpha=0.25)
        axis.set(xlim=(0.02, 0.28), title=f"{state.galaxy}, {ring}")
axes[0, 0].legend(frameon=False)
for axis in axes[-1]:
    axis.set_xlabel(r"$k$ (pixel$^{-1}$)")
for axis in axes[:, 0]:
    axis.set_ylabel("Нормированная форма")
fig.tight_layout()
fig.savefig(pilot_figure_dir / "sbf2_Ek_Nk_shapes.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)

template_shape_df = pd.DataFrame(template_rows)
display(template_shape_df)


/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/1387507047.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,galaxy,ring,E_N_correlation
0,NGC 3379,inner,0.990688
1,NGC 3379,outer,0.990471
2,NGC 1380,inner,0.982226
3,NGC 1380,outer,0.982531
4,NGC 4486,inner,0.997620
5,NGC 4486,outer,0.997460


## 6. Фиты спектра и замыкание на production

Для каждого кольца параллельно вычисляются классическая модель
$P_0E(k)+P_1$ и диагностическая модель $P_0E(k)+P_nN(k)$. В первой
таблице оставлены только принятые результаты 3.5σ; во второй — точность
воспроизведения production-ветви; в третьей — прямое сравнение моделей.
Ковариационные матрицы и служебные коэффициенты сохраняются в полных
DataFrame и CSV, но не засоряют вывод.

In [15]:
def fit_spectra(s):
    GALAXY = s.galaxy
    galaxy_tag, table_dir = s.galaxy_tag, s.table_dir
    prepared_rings, spectra = s.prepared_rings, s.spectra
    psf_ids, pr_by_ring = s.psf_ids, s.pr_by_ring
    ab_zeropoint = s.ab_zeropoint
    main_measurements, region_name = s.main_measurements, s.region_name

    fit_rows = []
    fit_residual_rows = []
    for ring, prepared in prepared_rings.items():
        effective_kmin = max(
            K_MIN,
            10.0 / min(prepared["shape"]),
        )
        k = prepared["k"]

        for variant, _ in WINSOR_VARIANTS:
            Pk = spectra[(ring, variant)]["Pk"]
            Pk_error = spectra[(ring, variant)]["Pk_error"]

            for psf_id, E_profile in zip(psf_ids, prepared["E_profiles"]):
                base_selected = (
                    (k >= effective_kmin)
                    & (k <= K_MAX)
                    & np.isfinite(Pk)
                    & np.isfinite(Pk_error)
                    & (Pk_error > 0)
                    & np.isfinite(E_profile)
                    & (E_profile > 0)
                )

                for noise_model in ["constant", "N(k)"]:
                    selected = base_selected.copy()
                    if noise_model == "N(k)":
                        selected &= (
                            np.isfinite(prepared["N_profile"])
                            & (prepared["N_profile"] > 0)
                        )
                    if (
                        noise_model == "N(k)"
                        and selected.sum() != base_selected.sum()
                    ):
                        raise RuntimeError(
                            f"{ring}: N(k) invalid inside fit range"
                        )
                    if int(selected.sum()) < 10:
                        continue

                    y = Pk[selected]
                    y_error = Pk_error[selected]
                    E = E_profile[selected]
                    second_template = (
                        np.ones_like(E)
                        if noise_model == "constant"
                        else prepared["N_profile"][selected]
                    )
                    fit = weighted_fit(
                        y,
                        y_error,
                        E,
                        second_template,
                    )
                    P0 = fit["P0"]
                    Pr = pr_by_ring[ring]
                    fluctuation_power = P0 - Pr
                    mbar = (
                        -2.5 * np.log10(fluctuation_power) + ab_zeropoint
                        if fluctuation_power > 0
                        else np.nan
                    )
                    mbar_sigma = (
                        (2.5 / np.log(10.0))
                        * fit["P0_sigma"]
                        / fluctuation_power
                        if fluctuation_power > 0
                        else np.nan
                    )

                    row = {
                        "galaxy": GALAXY,
                        "ring": ring,
                        "variant": variant,
                        "noise_model": noise_model,
                        "psf_id": psf_id,
                        "n_fit": int(selected.sum()),
                        "effective_kmin": effective_kmin,
                        "kmax": K_MAX,
                        "P0": P0,
                        "noise_coefficient": fit["noise_coefficient"],
                        "noise_coefficient_nonnegative": (
                            fit["noise_coefficient"] >= 0
                        ),
                        "physical_solution": bool(
                            fluctuation_power > 0
                            and (noise_model == "constant"
                                 or fit["noise_coefficient"] >= 0)
                        ),
                        "Pr": Pr,
                        "P_fluctuation": fluctuation_power,
                        "mbar_observed": mbar,
                        "mbar_sigma_formal": mbar_sigma,
                        "P0_sigma_formal": fit["P0_sigma"],
                        "cov_P0_P0": fit["cov_P0_P0"],
                        "cov_P0_Pnoise": fit["cov_P0_Pnoise"],
                        "cov_Pnoise_P0": fit["cov_Pnoise_P0"],
                        "cov_Pnoise_Pnoise": fit["cov_Pnoise_Pnoise"],
                        "chi2": fit["chi2"],
                        "chi2_reduced": fit["chi2_reduced"],
                        "aicc": fit["aicc"],
                        "condition_number": fit["condition_number"],
                        "template_correlation": fit["template_correlation"],
                        "covariance_correlation": fit["covariance_correlation"],
                    }
                    fit_rows.append(row)
                    for k_value, observed, predicted, residual_sigma in zip(
                        k[selected], y, fit["model_values"],
                        fit["standardized_residual"],
                    ):
                        fit_residual_rows.append({
                            "galaxy": GALAXY, "ring": ring,
                            "variant": variant, "noise_model": noise_model,
                            "psf_id": psf_id, "k": k_value,
                            "observed": observed, "predicted": predicted,
                            "residual": observed - predicted,
                            "residual_sigma": residual_sigma,
                        })

    fit_per_psf = pd.DataFrame(fit_rows)
    fit_residuals = pd.DataFrame(fit_residual_rows)
    fit_per_psf.to_csv(
        table_dir / f"{galaxy_tag}_noise_winsor_fit_per_psf.csv",
        index=False,
    )
    fit_residuals.to_csv(
        table_dir / f"{galaxy_tag}_fit_residuals.csv", index=False
    )

    summary_rows = []
    group_columns = ["galaxy", "ring", "variant", "noise_model"]
    for keys, group in fit_per_psf.groupby(group_columns, sort=False):
        P0_values = group["P0"].to_numpy(float)
        P0 = float(np.median(P0_values))
        P0_formal = float(np.median(group["P0_sigma_formal"]))
        P0_psf = float(1.4826 * np.median(np.abs(P0_values - P0)))
        P0_sigma = float(np.hypot(P0_formal, P0_psf))
        Pr = float(np.median(group["Pr"]))
        fluctuation_power = P0 - Pr
        mbar = (
            -2.5 * np.log10(fluctuation_power) + ab_zeropoint
            if fluctuation_power > 0 else np.nan
        )
        mbar_sigma = (
            (2.5 / np.log(10.0)) * P0_sigma / fluctuation_power
            if fluctuation_power > 0 else np.nan
        )

        summary_rows.append({
            **dict(zip(group_columns, keys)),
            "n_psf": len(group),
            "P0": P0,
            "P0_sigma_formal": P0_formal,
            "P0_psf_mad": P0_psf,
            "P0_sigma_total": P0_sigma,
            "noise_coefficient": float(
                np.median(group["noise_coefficient"])
            ),
            "noise_coefficient_negative_fraction": float(
                np.mean(group["noise_coefficient"] < 0)
            ),
            "P0_negative_fraction": float(np.mean(group["P0"] <= 0)),
            "physical_solution_fraction": float(
                np.mean(group["physical_solution"])
            ),
            "Pr": Pr,
            "P_fluctuation": fluctuation_power,
            "mbar_observed": mbar,
            "mbar_sigma_with_psf": mbar_sigma,
            "chi2": float(np.median(group["chi2"])),
            "chi2_reduced": float(np.median(group["chi2_reduced"])),
            "aicc": float(np.median(group["aicc"])),
            "condition_number": float(np.median(group["condition_number"])),
            "template_correlation": (
                float(group["template_correlation"].dropna().median())
                if group["template_correlation"].notna().any()
                else np.nan
            ),
            "covariance_correlation": float(
                np.nanmedian(group["covariance_correlation"])
            ),
        })

    fit_summary = pd.DataFrame(summary_rows)
    fit_summary["delta_P0_vs_raw"] = np.nan
    fit_summary["delta_P0_fraction_vs_raw"] = np.nan
    fit_summary["delta_mbar_vs_raw"] = np.nan
    fit_summary["delta_mbar_from_production_branch"] = np.nan

    for ring in s.rings:
        for noise_model in ["constant", "N(k)"]:
            group = (
                fit_summary["ring"].eq(ring)
                & fit_summary["noise_model"].eq(noise_model)
            )
            raw_rows = fit_summary[
                group & fit_summary["variant"].eq("raw")
            ]
            if raw_rows.empty:
                print(f"Нет raw-фита: {ring}, {noise_model}")
                continue
            raw = raw_rows.iloc[0]
            fit_summary.loc[group, "delta_P0_vs_raw"] = (
                fit_summary.loc[group, "P0"] - raw["P0"]
            )
            if raw["P0"] > 0:
                fit_summary.loc[group, "delta_P0_fraction_vs_raw"] = (
                    fit_summary.loc[group, "P0"] / raw["P0"] - 1.0
                )
            if np.isfinite(raw["mbar_observed"]):
                fit_summary.loc[group, "delta_mbar_vs_raw"] = (
                    fit_summary.loc[group, "mbar_observed"]
                    - raw["mbar_observed"]
                )

        production_rows = fit_summary[
            fit_summary["ring"].eq(ring)
            & fit_summary["variant"].eq("winsor_3.5")
            & fit_summary["noise_model"].eq("constant")
        ]
        if production_rows.empty:
            print(f"Нет production-ветви: {ring}")
            continue
        production_branch = production_rows.iloc[0]
        fit_summary.loc[
            fit_summary["ring"].eq(ring),
            "delta_mbar_from_production_branch",
        ] = (
            fit_summary.loc[
                fit_summary["ring"].eq(ring), "mbar_observed"
            ] - production_branch["mbar_observed"]
        )

    fit_summary.to_csv(
        table_dir / f"{galaxy_tag}_noise_winsor_fit_summary.csv",
        index=False,
    )

    closure_rows = []

    for ring, prepared in prepared_rings.items():
        production = main_measurements[
            main_measurements["region"].eq(region_name[ring])
        ].iloc[0]
        recreated = fit_summary[
            fit_summary["ring"].eq(ring)
            & fit_summary["variant"].eq("winsor_3.5")
            & fit_summary["noise_model"].eq("constant")
        ].iloc[0]

        delta_mbar = (
            float(recreated["mbar_observed"])
            - float(production["mbar_spec"])
        )
        relative_P0 = (
            float(recreated["P0"]) / float(production["P0"]) - 1.0
        )
        relative_P1 = (
            (float(recreated["noise_coefficient"]) - float(production["P1"]))
            / max(abs(float(production["P1"])), 1e-12)
        )
        relative_Imean = (
            prepared["mean_model"] / float(production["Imean"]) - 1.0
        )
        same_n_use = (
            int(prepared["n_use"]) == int(production["n_use"])
        )
        same_crop = (
            prepared["shape"][0] == int(production["fft_crop_ny"])
            and prepared["shape"][1] == int(production["fft_crop_nx"])
        )
        passed = (
            abs(delta_mbar) <= REPRO_MAG_TOL
            and abs(relative_P0) <= REPRO_POWER_REL_TOL
            and abs(relative_P1) <= REPRO_POWER_REL_TOL
            and abs(relative_Imean) <= 1e-6
            and same_n_use
            and same_crop
        )

        closure_rows.append({
            "galaxy": GALAXY,
            "ring": ring,
            "production_P0": float(production["P0"]),
            "recreated_P0": float(recreated["P0"]),
            "relative_P0_difference": relative_P0,
            "production_P1": float(production["P1"]),
            "recreated_P1": float(recreated["noise_coefficient"]),
            "relative_P1_difference": relative_P1,
            "production_Imean": float(production["Imean"]),
            "recreated_Imean": prepared["mean_model"],
            "relative_Imean_difference": relative_Imean,
            "production_mbar": float(production["mbar_spec"]),
            "recreated_mbar": float(recreated["mbar_observed"]),
            "delta_mbar": delta_mbar,
            "production_n_use": int(production["n_use"]),
            "recreated_n_use": int(prepared["n_use"]),
            "same_n_use": same_n_use,
            "same_crop": same_crop,
            "passed": passed,
        })

    baseline_closure = pd.DataFrame(closure_rows)
    baseline_closure.to_csv(
        table_dir / f"{galaxy_tag}_reproduction_check.csv",
        index=False,
    )

    if not baseline_closure["passed"].all():
        raise RuntimeError(
            "Production-ветвь не воспроизведена; "
            "новые сдвиги интерпретировать нельзя"
        )
    print("Production-ветвь воспроизведена.")

    factorial_rows = []

    for ring in s.rings:
        selected = fit_summary[fit_summary["ring"].eq(ring)].set_index(
            ["variant", "noise_model"]
        )

        raw_constant = selected.loc[
            ("raw", "constant"), "mbar_observed"
        ]
        winsor_constant = selected.loc[
            ("winsor_3.5", "constant"), "mbar_observed"
        ]
        raw_noise = selected.loc[
            ("raw", "N(k)"), "mbar_observed"
        ]
        winsor_noise = selected.loc[
            ("winsor_3.5", "N(k)"), "mbar_observed"
        ]

        factorial_rows.append({
            "galaxy": GALAXY,
            "ring": ring,
            "winsor_effect_with_constant_noise": (
                winsor_constant - raw_constant
            ),
            "winsor_effect_with_Nk": (
                winsor_noise - raw_noise
            ),
            "Nk_effect_on_raw": (
                raw_noise - raw_constant
            ),
            "Nk_effect_on_winsor_3p5": (
                winsor_noise - winsor_constant
            ),
            "interaction": (
                (winsor_noise - winsor_constant)
                - (raw_noise - raw_constant)
            ),
        })

    factorial_effects = pd.DataFrame(factorial_rows)
    factorial_effects.to_csv(
        table_dir / f"{galaxy_tag}_noise_winsor_factorial_effects.csv",
        index=False,
    )

    classic = fit_summary[
        fit_summary["noise_model"].eq("constant")
    ].rename(columns={
        "mbar_observed": "mbar_constant",
        "aicc": "aicc_constant",
    })
    noise_fit = fit_summary[
        fit_summary["noise_model"].eq("N(k)")
    ].rename(columns={
        "mbar_observed": "mbar_Nk",
        "aicc": "aicc_Nk",
        "noise_coefficient": "Pn",
        "template_correlation": "E_N_correlation",
        "condition_number": "E_N_condition_number",
    })

    model_comparison = classic[[
        "galaxy", "ring", "variant",
        "mbar_constant", "aicc_constant",
    ]].merge(
        noise_fit[[
            "galaxy", "ring", "variant",
            "mbar_Nk", "aicc_Nk", "Pn",
            "E_N_correlation", "E_N_condition_number",
        ]],
        on=["galaxy", "ring", "variant"],
        validate="one_to_one",
    )
    model_comparison["delta_mbar_Nk_minus_constant"] = (
        model_comparison["mbar_Nk"]
        - model_comparison["mbar_constant"]
    )
    model_comparison["delta_aicc_Nk_minus_constant"] = (
        model_comparison["aicc_Nk"]
        - model_comparison["aicc_constant"]
    )
    model_comparison["Pn_nonnegative"] = model_comparison["Pn"] >= 0
    model_comparison.to_csv(
        table_dir / f"{galaxy_tag}_noise_model_comparison.csv",
        index=False,
    )


    s.fit_summary = fit_summary
    s.fit_residuals = fit_residuals
    s.model_comparison = model_comparison
    s.baseline_closure = baseline_closure

In [16]:

for state in states:
    fit_spectra(state)

fit_summary_df = pd.concat(
    [state.fit_summary for state in states], ignore_index=True
)
fit_residuals_df = pd.concat(
    [state.fit_residuals for state in states], ignore_index=True
)
model_comparison_df = pd.concat(
    [state.model_comparison for state in states], ignore_index=True
)
closure_df = pd.concat(
    [state.baseline_closure for state in states], ignore_index=True
)

adopted_fit_view = fit_summary_df[
    fit_summary_df["variant"].eq("winsor_3.5")
][[
    "galaxy", "ring", "noise_model", "P0", "noise_coefficient",
    "mbar_observed", "mbar_sigma_with_psf", "chi2_reduced",
    "aicc", "physical_solution_fraction",
]]
model_view = model_comparison_df[
    model_comparison_df["variant"].eq("winsor_3.5")
][[
    "galaxy", "ring", "mbar_constant", "mbar_Nk",
    "delta_aicc_Nk_minus_constant", "E_N_correlation",
    "Pn_nonnegative",
]]

display(adopted_fit_view)
display(closure_df[[
    "galaxy", "ring", "delta_mbar", "same_n_use", "same_crop", "passed",
]])
display(model_view)


Production-ветвь воспроизведена.
Production-ветвь воспроизведена.
Production-ветвь воспроизведена.


,galaxy,ring,noise_model,P0,noise_coefficient,mbar_observed,mbar_sigma_with_psf,chi2_reduced,aicc,physical_solution_fraction
4,NGC 3379,inner,constant,2.669611,-0.232776,26.933681,0.020244,29.071431,644.142904,1.0
5,NGC 3379,inner,N(k),-0.718692,0.682984,NaN,NaN,803.042058,17671.496701,0.2
12,NGC 3379,outer,constant,2.302247,-0.172240,27.101682,0.012630,30.074337,666.206844,1.0
13,NGC 3379,outer,N(k),-0.750294,0.668494,NaN,NaN,1460.907035,32144.526189,0.0
20,NGC 1380,inner,constant,0.916718,-0.078828,28.094008,0.021230,30.789732,681.945530,1.0
21,NGC 1380,inner,N(k),0.526926,-0.000622,28.695241,1.396958,773.296248,17017.088885,0.4
28,NGC 1380,outer,constant,0.879991,-0.072778,28.139850,0.018766,73.686670,1625.678162,1.0
29,NGC 1380,outer,N(k),0.511826,0.001969,28.729300,1.288615,2206.696083,48551.885264,0.6
36,NGC 4486,inner,constant,1.078096,-0.090276,27.920484,0.023444,37.966176,839.827294,1.0
37,NGC 4486,inner,N(k),-1.219290,0.616755,NaN,NaN,48.242349,1065.903096,0.0


,galaxy,ring,delta_mbar,same_n_use,same_crop,passed
0,NGC 3379,inner,1.191143e-08,True,True,True
1,NGC 3379,outer,-6.255618e-09,True,True,True
2,NGC 1380,inner,1.192223e-08,True,True,True
3,NGC 1380,outer,7.476402e-09,True,True,True
4,NGC 4486,inner,-1.383523e-09,True,True,True
5,NGC 4486,outer,-1.113570e-09,True,True,True


,galaxy,ring,mbar_constant,mbar_Nk,delta_aicc_Nk_minus_constant,E_N_correlation,Pn_nonnegative
2,NGC 3379,inner,26.933681,NaN,17027.353797,0.990662,True
6,NGC 3379,outer,27.101682,NaN,31478.319345,0.990400,True
10,NGC 1380,inner,28.094008,28.695241,16335.143355,0.982189,False
14,NGC 1380,outer,28.139850,28.729300,46926.207103,0.982573,True
18,NGC 4486,inner,27.920484,NaN,226.075802,0.997449,True
22,NGC 4486,outer,27.851802,NaN,-226.762197,0.997439,True


In [17]:
def plot_fit_diagnostics(s):
    GALAXY = s.galaxy
    galaxy_tag, figure_dir = s.galaxy_tag, s.figure_dir
    prepared_rings, spectra = s.prepared_rings, s.spectra
    fit_summary, fit_residuals = s.fit_summary, s.fit_residuals

    variant_to_plot = "winsor_3.5"
    fig, axes = plt.subplots(
        2, 2, figsize=(10, 7), sharex="col",
        gridspec_kw={"height_ratios": [2.2, 1]},
    )

    for column, ring in enumerate(["inner", "outer"]):
        spectrum = spectra[(ring, variant_to_plot)]
        lines = fit_residuals[
            fit_residuals["ring"].eq(ring)
            & fit_residuals["variant"].eq(variant_to_plot)
        ]
        classic = (
            lines[lines["noise_model"].eq("constant")]
            .groupby("k", as_index=False)
            .agg(predicted=("predicted", "median"),
                 residual_sigma=("residual_sigma", "median"))
        )
        noise = (
            lines[lines["noise_model"].eq("N(k)")]
            .groupby("k", as_index=False)
            .agg(predicted=("predicted", "median"),
                 residual_sigma=("residual_sigma", "median"))
        )
        shown = classic.merge(noise, on="k", suffixes=("_constant", "_Nk"))
        k = prepared_rings[ring]["k"]
        observed = pd.DataFrame({
            "k": k,
            "Pk": spectrum["Pk"],
            "Pk_error": spectrum["Pk_error"],
        })
        shown = shown.merge(observed, on="k", validate="one_to_one")

        axes[0, column].errorbar(
            shown["k"], shown["Pk"], yerr=shown["Pk_error"],
            fmt=".", color="black", ms=4, lw=0.7,
        )
        axes[0, column].plot(
            shown["k"], shown["predicted_constant"],
            color="#dc2626", lw=2, label=r"$P_0E+P_1$",
        )
        axes[0, column].plot(
            shown["k"], shown["predicted_Nk"],
            color="#2563eb", lw=2, label=r"$P_0E+P_nN$",
        )
        axes[1, column].axhline(0, color="0.4", lw=1)
        axes[1, column].plot(
            shown["k"], shown["residual_sigma_constant"],
            color="#dc2626", lw=1.2,
        )
        axes[1, column].plot(
            shown["k"], shown["residual_sigma_Nk"],
            color="#2563eb", lw=1.2,
        )
        axes[0, column].set_title(
            "Внутреннее кольцо" if ring == "inner" else "Внешнее кольцо"
        )
        axes[1, column].set_xlabel(r"$k$ (pixel$^{-1}$)")

    axes[0, 0].set_ylabel(r"$P(k)$")
    axes[1, 0].set_ylabel("Остаток / ошибка")
    axes[0, 0].legend(frameon=False)
    fig.suptitle(GALAXY)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    fig.savefig(
        figure_dir / f"{galaxy_tag}_noise_model_fits.pdf",
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

    variant_order = [item[0] for item in WINSOR_VARIANTS]
    x = np.arange(len(variant_order))
    fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharey=True)

    for axis, ring in zip(axes, ["inner", "outer"]):
        for noise_model, style, color in [
            ("constant", "-", "#dc2626"),
            ("N(k)", "--", "#2563eb"),
        ]:
            values = (
                fit_summary[
                    fit_summary["ring"].eq(ring)
                    & fit_summary["noise_model"].eq(noise_model)
                ]
                .set_index("variant")
                .loc[variant_order, "mbar_observed"]
            )
            axis.plot(
                x,
                values,
                marker="o",
                ls=style,
                color=color,
                label=noise_model,
            )

        axis.set_xticks(x, ["raw", "3.0", "3.5", "4.0"])
        axis.set_xlabel("Предел винзорирования")
        axis.set_title(
            "Внутреннее кольцо" if ring == "inner" else "Внешнее кольцо"
        )

    axes[0].invert_yaxis()

    axes[0].set_ylabel(r"$\bar m_{150}$ (mag)")
    axes[0].legend(frameon=False)
    fig.suptitle(GALAXY)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    fig.savefig(
        figure_dir / f"{galaxy_tag}_winsor_sensitivity.pdf",
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

### Как читать фиты и чувствительность к винзорированию

Для каждой галактики сначала показаны измеренный $P(k)$, две модели и их
остатки в единицах ошибки. Желательны беспорядочные остатки около нуля.
Меньший AICc не спасает модель, если $P_0 ≤ P_r$: такая SBF-амплитуда
нефизична.

Второй рисунок показывает изменение m̄ при raw, 3.0σ, 3.5σ и 4.0σ.
Плоская последовательность означает устойчивость. Если синяя линия
$N(k)$ отсутствует, это NaN из-за нефизического $P_0$, а не ошибка
графика.

In [18]:

for state in states:
    plot_fit_diagnostics(state)


/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/236747585.py:75: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/236747585.py:120: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/236747585.py:75: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/236747585.py:120: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/236747585.py:75: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/236747585.py:120: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Power-spectrum fits (English figure set)

The black points are the mean measured two-dimensional Fourier power in
radial frequency bins. Their vertical error bars are the standard errors of
the Fourier modes contributing to each bin (SEM). The red curve is the
classical $P(k)=P_0E(k)+P_1$ fit; the blue curve is the diagnostic
$P(k)=P_0E(k)+P_nN(k)$ fit. The lower panels show residuals divided by the
same bin SEM. The spectra, fitting range, masks, and $3.5\sigma$
winsorization are unchanged from the Russian figures above.


In [19]:
variant_to_plot = "winsor_3.5"

for state in states:
    fig, axes = plt.subplots(
        2, 2, figsize=(10, 7), sharex="col",
        gridspec_kw={"height_ratios": [2.2, 1]},
    )

    for column, ring in enumerate(["inner", "outer"]):
        spectrum = state.spectra[(ring, variant_to_plot)]
        lines = state.fit_residuals[
            state.fit_residuals["ring"].eq(ring)
            & state.fit_residuals["variant"].eq(variant_to_plot)
        ]
        classic = (
            lines[lines["noise_model"].eq("constant")]
            .groupby("k", as_index=False)
            .agg(predicted=("predicted", "median"),
                 residual_sigma=("residual_sigma", "median"))
        )
        correlated = (
            lines[lines["noise_model"].eq("N(k)")]
            .groupby("k", as_index=False)
            .agg(predicted=("predicted", "median"),
                 residual_sigma=("residual_sigma", "median"))
        )
        shown = classic.merge(
            correlated, on="k", suffixes=("_constant", "_Nk")
        )
        prepared = state.prepared_rings[ring]
        observed = pd.DataFrame({
            "k": prepared["k"],
            "Pk": spectrum["Pk"],
            "Pk_error": spectrum["Pk_error"],
        })
        shown = shown.merge(observed, on="k", validate="one_to_one")

        axes[0, column].errorbar(
            shown["k"], shown["Pk"], yerr=shown["Pk_error"],
            fmt=".", color="black", ms=4, lw=0.7,
            label=r"Measured radial-bin mean ($\pm$ SEM)",
        )
        axes[0, column].plot(
            shown["k"], shown["predicted_constant"],
            color="#dc2626", lw=2, label=r"$P_0E(k)+P_1$",
        )
        axes[0, column].plot(
            shown["k"], shown["predicted_Nk"],
            color="#2563eb", lw=2, label=r"$P_0E(k)+P_nN(k)$",
        )
        axes[1, column].axhline(0, color="0.4", lw=1)
        axes[1, column].plot(
            shown["k"], shown["residual_sigma_constant"],
            color="#dc2626", lw=1.2,
        )
        axes[1, column].plot(
            shown["k"], shown["residual_sigma_Nk"],
            color="#2563eb", lw=1.2,
        )
        axes[0, column].set_title(
            "Inner annulus" if ring == "inner" else "Outer annulus"
        )
        axes[1, column].set_xlabel(r"$k$ (pixel$^{-1}$)")

    axes[0, 0].set_ylabel(r"$P(k)$")
    axes[1, 0].set_ylabel("Residual / bin SEM")
    axes[0, 0].legend(frameon=False, fontsize=9)
    fig.suptitle(f"{state.galaxy}: power-spectrum fits")
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    fig.savefig(
        state.figure_dir
        / f"{state.galaxy_tag}_noise_model_fits_english.pdf",
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)


/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/1984873863.py:75: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/1984873863.py:75: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/1984873863.py:75: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Чувствительность к размеру PSF

Тест повторяет оконный FFT-фит с PSF размером 129 и 257 пикселей.
Техническая информация STPSF, включая `opd_corr_id`, остаётся в
provenance-CSV и FITS-заголовках, но на экран не выводится.

In [20]:
def test_psf_size(s):
    import stpsf

    GALAXY = s.galaxy
    galaxy_tag = s.galaxy_tag
    validation_root = s.validation_root
    table_dir, figure_dir = s.table_dir, s.figure_dir
    run_dir, stem = s.run_dir, s.stem
    signal_path = s.signal_path
    psfs = s.psfs
    prepared_rings, spectra = s.prepared_rings, s.spectra
    pr_by_ring, ab_zeropoint = s.pr_by_ring, s.ab_zeropoint

    psf_library_path = run_dir / f"{stem}_psf_library.csv"
    psf_library = pd.read_csv(psf_library_path)
    nominal_meta = psf_library.iloc[0].copy()

    psf_output_dir = validation_root / "psf"
    psf_output_dir.mkdir(exist_ok=True)
    psf_257_path = psf_output_dir / f"{galaxy_tag}_F150W_psf_257_normalized.fits"
    legacy_cache = (
        project_root / "runs" / "sbf2_go3055" / "analysis"
        / "psf_257_cache" / f"{GALAXY.replace(' ', '_')}_F150W_psf_257.fits"
    )

    detector_position = [
        int(float(value.strip()))
        for value in str(nominal_meta["detector_position"]).strip("()[]").split(",")
    ]
    opd_path = Path(nominal_meta["opd_path"])
    selected_extension = str(nominal_meta["selected_extension"])

    if psf_257_path.is_file():
        with fits.open(psf_257_path, memmap=False) as hdul:
            psf_257_raw = np.asarray(hdul[0].data, dtype=float)
            raw_sum_257 = float(hdul[0].header.get("RAWSUM", psf_257_raw.sum()))
        generation_mode = "normalized project cache"
    elif legacy_cache.is_file():
        psf_257_raw = np.asarray(fits.getdata(legacy_cache), dtype=float)
        raw_sum_257 = float(psf_257_raw.sum())
        generation_mode = "legacy cache, metadata restored here"
    else:
        stpsf_data_dir = Path(
            os.environ.get("STPSF_PATH", Path.home() / "data" / "stpsf-data")
        ).expanduser().resolve()
        if not stpsf_data_dir.is_dir():
            raise FileNotFoundError(f"Нет локальных данных STPSF: {stpsf_data_dir}")
        os.environ["STPSF_PATH"] = str(stpsf_data_dir)

        primary_header = fits.getheader(signal_path, 0)
        simulator = stpsf.instrument(primary_header["INSTRUME"])
        simulator.filter = str(nominal_meta["filter"])
        simulator.set_position_from_aperture_name(str(nominal_meta["aperture"]))
        simulator.detector_position = tuple(detector_position)

        opd_relative = Path(os.path.relpath(opd_path.resolve(), Path.cwd()))
        opd_candidates = [opd_relative, opd_path.resolve()]
        opd_argument = next(
            str(candidate)
            for candidate in opd_candidates
            if candidate.is_file() and str(candidate).isascii()
        )
        simulator.load_wss_opd(opd_argument, verbose=False, plot=False)
        simulator.options["output_mode"] = "both"
        product = simulator.calc_psf(
            source=None,
            nlambda=PSF_NLAMBDA,
            fov_pixels=257,
            fft_oversample=PSF_FFT_OVERSAMPLE,
            detector_oversample=PSF_DETECTOR_OVERSAMPLE,
            add_distortion=PSF_ADD_DISTORTION,
        )
        for extension in ["DET_DIST", "DET_SAMP"]:
            if extension in product:
                selected_extension = extension
                psf_257_raw = np.asarray(product[extension].data, dtype=float)
                break
        else:
            raise RuntimeError("STPSF не вернул detector-sampled PSF")
        raw_sum_257 = float(psf_257_raw.sum())
        generation_mode = "built offline from local STPSF data"

    if psf_257_raw.shape != (257, 257):
        raise RuntimeError(f"Ожидалась PSF 257x257, получено {psf_257_raw.shape}")
    normalization_sum_257 = float(np.nansum(psf_257_raw))
    if not np.isfinite(normalization_sum_257) or normalization_sum_257 <= 0:
        raise RuntimeError(
            f"Некорректный интеграл PSF 257: {normalization_sum_257}"
        )

    psf_257 = (
        np.nan_to_num(psf_257_raw, nan=0.0) / normalization_sum_257
    )

    psf_header = fits.Header()
    psf_header["GALAXY"] = GALAXY
    psf_header["FILTER"] = str(nominal_meta["filter"])
    psf_header["APERNAME"] = str(nominal_meta["aperture"])
    psf_header["OPDCORR"] = str(nominal_meta["opd_corr_id"])
    psf_header["OPDDAYS"] = float(nominal_meta["opd_delta_days"])
    psf_header["DETX"] = detector_position[0]
    psf_header["DETY"] = detector_position[1]
    psf_header["NLAMBDA"] = PSF_NLAMBDA
    psf_header["FFTOVER"] = PSF_FFT_OVERSAMPLE
    psf_header["DETOVER"] = PSF_DETECTOR_OVERSAMPLE
    psf_header["DISTORT"] = PSF_ADD_DISTORTION
    psf_header["STAMPSZ"] = 257
    psf_header["RAWSUM"] = raw_sum_257
    psf_header["PSFSUM"] = float(psf_257.sum())
    psf_header["SRCARG"] = PSF_SOURCE_ARGUMENT
    psf_header["SPECTRUM"] = PSF_SOURCE_SPECTRUM
    psf_header["STPSFVER"] = package_version("stpsf")
    psf_header["POPPYVER"] = package_version("poppy")
    psf_header["SYNPHOT"] = package_version("synphot")
    psf_header["EXTUSED"] = selected_extension
    psf_header["NORM"] = "unit integral after extension selection"
    fits.writeto(
        psf_257_path,
        psf_257.astype("float32"),
        header=psf_header,
        overwrite=True,
    )

    psf_metadata = psf_library.copy()
    psf_metadata["requested_fov_pixels"] = 129
    psf_metadata["nlambda"] = PSF_NLAMBDA
    psf_metadata["fft_oversample"] = PSF_FFT_OVERSAMPLE
    psf_metadata["detector_oversample"] = PSF_DETECTOR_OVERSAMPLE
    psf_metadata["add_distortion"] = PSF_ADD_DISTORTION
    psf_metadata["source_argument"] = PSF_SOURCE_ARGUMENT
    psf_metadata["source_spectrum"] = PSF_SOURCE_SPECTRUM
    psf_metadata["normalization"] = "unit integral after extension selection"
    psf_metadata["stpsf_version"] = package_version("stpsf")
    psf_metadata["poppy_version"] = package_version("poppy")
    psf_metadata["synphot_version"] = package_version("synphot")
    psf_metadata["generation_mode"] = "production cache"
    psf_metadata["raw_sum_before_normalization"] = np.nan

    row_257 = nominal_meta.to_dict()
    row_257.update({
        "psf_id": f"{nominal_meta['psf_id']}_stamp257",
        "kind": "model_stamp_size_test",
        "selected_extension": selected_extension,
        "shape": str(psf_257.shape),
        "sum": float(psf_257.sum()),
        "requested_fov_pixels": 257,
        "nlambda": PSF_NLAMBDA,
        "fft_oversample": PSF_FFT_OVERSAMPLE,
        "detector_oversample": PSF_DETECTOR_OVERSAMPLE,
        "add_distortion": PSF_ADD_DISTORTION,
        "source_argument": PSF_SOURCE_ARGUMENT,
        "source_spectrum": PSF_SOURCE_SPECTRUM,
        "normalization": "unit integral after extension selection",
        "stpsf_version": package_version("stpsf"),
        "poppy_version": package_version("poppy"),
        "synphot_version": package_version("synphot"),
        "generation_mode": generation_mode,
        "raw_sum_before_normalization": raw_sum_257,
    })
    psf_metadata = pd.concat(
        [psf_metadata, pd.DataFrame([row_257])],
        ignore_index=True,
    )
    psf_metadata.to_csv(
        table_dir / f"{galaxy_tag}_psf_normalization_metadata.csv",
        index=False,
    )

    ee_rows = []
    for stamp_size, image in [(129, psfs[0]), (257, psf_257)]:
        image = np.asarray(image, dtype=float)
        image /= image.sum()
        yy, xx = np.indices(image.shape)
        radius = np.hypot(
            yy - (image.shape[0] - 1) / 2,
            xx - (image.shape[1] - 1) / 2,
        )
        for aperture_radius in [1, 2, 3, 5, 10, 20, 32, 48, 64, 96, 128]:
            if aperture_radius <= image.shape[0] // 2:
                ee_rows.append({
                    "galaxy": GALAXY,
                    "psf_size_pix": stamp_size,
                    "radius_pix": aperture_radius,
                    "encircled_energy": float(image[radius <= aperture_radius].sum()),
                })

    center_257 = psf_257.shape[0] // 2
    central_129 = psf_257[
        center_257 - 64:center_257 + 65,
        center_257 - 64:center_257 + 65,
    ]
    fraction_129_within_257 = float(central_129.sum())
    psf_flux_summary = pd.DataFrame([{
        "galaxy": GALAXY,
        "fraction_129_within_257": fraction_129_within_257,
        "missing_outside_129_percent": 100 * (1 - fraction_129_within_257),
        "sum_psf_257_normalized": float(psf_257.sum()),
        "sum_production_psf_129_normalized": float(psfs[0].sum()),
    }])
    psf_encircled_energy = pd.DataFrame(ee_rows)
    psf_encircled_energy.to_csv(
        table_dir / f"{galaxy_tag}_psf_encircled_energy.csv",
        index=False,
    )
    psf_flux_summary.to_csv(
        table_dir / f"{galaxy_tag}_psf_129_257_flux_summary.csv",
        index=False,
    )

    psf_expectation = {}

    for ring, prepared in prepared_rings.items():
        shape = prepared["shape"]
        if min(shape) < 257:
            raise RuntimeError(f"{ring}: FFT-crop {shape} меньше PSF 257")

        padded_psf = np.zeros(shape, dtype=float)
        y0 = shape[0] // 2 - psf_257.shape[0] // 2
        x0 = shape[1] // 2 - psf_257.shape[1] // 2
        padded_psf[
            y0:y0 + psf_257.shape[0],
            x0:x0 + psf_257.shape[1],
        ] = psf_257

        with set_workers(FFT_WORKERS):
            profile_257 = monte_carlo_template(
                prepared["window"],
                prepared["radial_plan"],
                fft2(padded_psf),
                E_REALIZATIONS,
                RANDOM_SEED,
            )

        psf_expectation[ring] = {
            129: prepared["E_profiles"][0],
            257: profile_257,
        }

    psf_fit_rows = []
    for ring, prepared in prepared_rings.items():
        effective_kmin = max(K_MIN, 10.0 / min(prepared["shape"]))
        k = prepared["k"]

        for variant, _ in WINSOR_VARIANTS:
            Pk = spectra[(ring, variant)]["Pk"]
            Pk_error = spectra[(ring, variant)]["Pk_error"]

            for stamp_size in [129, 257]:
                E_profile = psf_expectation[ring][stamp_size]
                base_selected = (
                    (k >= effective_kmin)
                    & (k <= K_MAX)
                    & np.isfinite(Pk)
                    & np.isfinite(Pk_error)
                    & (Pk_error > 0)
                    & np.isfinite(E_profile)
                    & (E_profile > 0)
                )

                for noise_model in ["constant", "N(k)"]:
                    selected = base_selected.copy()
                    if noise_model == "N(k)":
                        selected &= (
                            np.isfinite(prepared["N_profile"])
                            & (prepared["N_profile"] > 0)
                        )
                        second_template = prepared["N_profile"][selected]
                    else:
                        second_template = np.ones(int(selected.sum()), dtype=float)

                    if int(selected.sum()) < 10:
                        continue

                    fit = weighted_fit(
                        Pk[selected],
                        Pk_error[selected],
                        E_profile[selected],
                        second_template,
                    )
                    P0 = fit["P0"]
                    fluctuation_power = P0 - pr_by_ring[ring]
                    mbar = (
                        -2.5 * np.log10(fluctuation_power) + ab_zeropoint
                        if fluctuation_power > 0 else np.nan
                    )
                    mbar_sigma = (
                        (2.5 / np.log(10.0))
                        * fit["P0_sigma"] / fluctuation_power
                        if fluctuation_power > 0 else np.nan
                    )

                    psf_fit_rows.append({
                        "galaxy": GALAXY,
                        "ring": ring,
                        "variant": variant,
                        "noise_model": noise_model,
                        "psf_size_pix": stamp_size,
                        "P0": P0,
                        "noise_coefficient": fit["noise_coefficient"],
                        "physical_solution": bool(
                            fluctuation_power > 0
                            and (
                                noise_model == "constant"
                                or fit["noise_coefficient"] >= 0
                            )
                        ),
                        "Pr": pr_by_ring[ring],
                        "P_fluctuation": fluctuation_power,
                        "mbar_observed": mbar,
                        "mbar_sigma_formal": mbar_sigma,
                        "chi2": fit["chi2"],
                        "chi2_reduced": fit["chi2_reduced"],
                        "aicc": fit["aicc"],
                        "condition_number": fit["condition_number"],
                        "template_correlation": fit["template_correlation"],
                        "covariance_correlation": fit["covariance_correlation"],
                        "cov_P0_P0": fit["cov_P0_P0"],
                        "cov_P0_Pnoise": fit["cov_P0_Pnoise"],
                        "cov_Pnoise_P0": fit["cov_Pnoise_P0"],
                        "cov_Pnoise_Pnoise": fit["cov_Pnoise_Pnoise"],
                    })

    psf_size_fits = pd.DataFrame(psf_fit_rows)
    psf_size_fits.to_csv(
        table_dir / f"{galaxy_tag}_psf_129_257_fits.csv",
        index=False,
    )

    comparison_keys = ["galaxy", "ring", "variant", "noise_model"]
    fit_129 = psf_size_fits[
        psf_size_fits["psf_size_pix"].eq(129)
    ].rename(columns={
        "P0": "P0_129",
        "mbar_observed": "mbar_129",
        "aicc": "aicc_129",
        "physical_solution": "physical_129",
    })
    fit_257 = psf_size_fits[
        psf_size_fits["psf_size_pix"].eq(257)
    ].rename(columns={
        "P0": "P0_257",
        "mbar_observed": "mbar_257",
        "aicc": "aicc_257",
        "physical_solution": "physical_257",
    })
    psf_size_comparison = fit_129[
        comparison_keys + [
            "P0_129", "mbar_129", "aicc_129", "physical_129",
        ]
    ].merge(
        fit_257[
            comparison_keys + [
                "P0_257", "mbar_257", "aicc_257", "physical_257",
            ]
        ],
        on=comparison_keys,
        validate="one_to_one",
    )
    psf_size_comparison["delta_P0_257_minus_129"] = (
        psf_size_comparison["P0_257"] - psf_size_comparison["P0_129"]
    )
    psf_size_comparison["delta_mbar_257_minus_129"] = (
        psf_size_comparison["mbar_257"] - psf_size_comparison["mbar_129"]
    )
    psf_size_comparison.to_csv(
        table_dir / f"{galaxy_tag}_psf_129_257_fit_comparison.csv",
        index=False,
    )

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    for stamp_size, group in psf_encircled_energy.groupby("psf_size_pix"):
        axes[0].plot(
            group["radius_pix"],
            group["encircled_energy"],
            marker="o",
            label=f"{stamp_size} пикс.",
        )
    axes[0].set(
        xlabel="Радиус (пиксели)",
        ylabel="Доля потока внутри радиуса",
        xscale="log",
        title="Нормировка PSF",
    )
    axes[0].legend(frameon=False)

    classic_comparison = psf_size_comparison[
        psf_size_comparison["noise_model"].eq("constant")
    ]
    for ring, marker, label in [
        ("inner", "o", "внутреннее"),
        ("outer", "s", "внешнее"),
    ]:
        group = (
            classic_comparison[classic_comparison["ring"].eq(ring)]
            .set_index("variant")
            .loc[[item[0] for item in WINSOR_VARIANTS]]
        )
        axes[1].plot(
            np.arange(len(group)),
            group["delta_mbar_257_minus_129"],
            marker=marker,
            label=label,
        )
    axes[1].axhline(0, color="0.4", lw=1)
    axes[1].set_xticks(np.arange(4), ["raw", "3.0", "3.5", "4.0"])
    axes[1].set(
        xlabel="Винзорирование",
        ylabel=r"$\bar m_{257}-\bar m_{129}$ (mag)",
        title="Повторный оконный FFT-фит",
    )
    axes[1].legend(frameon=False)

    fig.suptitle(GALAXY)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    fig.savefig(
        figure_dir / f"{galaxy_tag}_psf_129_257_sensitivity.pdf",
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)


    s.psf_metadata = psf_metadata
    s.psf_encircled_energy = psf_encircled_energy
    s.psf_flux_summary = psf_flux_summary
    s.psf_size_comparison = psf_size_comparison


### Как читать графики PSF

Слева на каждом рисунке показана накопленная доля потока PSF, справа — сдвиг
m̄ при замене штампа 129 на 257 пикселей. Общий знак и близкая величина
сдвига у трёх объектов означают систематику нуль-пункта, которая не
усредняется как случайная ошибка. Следующая ячейка выводит графики всех трёх
галактик и одну общую компактную таблицу.

In [21]:

for state in states:
    test_psf_size(state)

psf_flux_df = pd.concat(
    [state.psf_flux_summary for state in states], ignore_index=True
)
psf_encircled_energy_df = pd.concat(
    [state.psf_encircled_energy for state in states], ignore_index=True
)
psf_comparison_df = pd.concat(
    [state.psf_size_comparison for state in states], ignore_index=True
)

adopted_psf_view = psf_comparison_df[
    psf_comparison_df["variant"].eq("winsor_3.5")
    & psf_comparison_df["noise_model"].eq("constant")
][[
    "galaxy", "ring", "mbar_129", "mbar_257",
    "delta_mbar_257_minus_129", "physical_129", "physical_257",
]]
display(psf_flux_df[["galaxy", "missing_outside_129_percent"]])
display(adopted_psf_view)


**WARNING**: LOCAL JWST PRD VERSION PRDOPSSOC-072 DOESN'T MATCH THE CURRENT ONLINE VERSION PRDOPSSOC-073
Please consider updating pysiaf, e.g. pip install --upgrade pysiaf or conda update pysiaf


/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/2153465197.py:420: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/2153465197.py:420: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/2153465197.py:420: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,galaxy,missing_outside_129_percent
0,NGC 3379,0.692035
1,NGC 1380,0.692032
2,NGC 4486,0.694765


,galaxy,ring,mbar_129,mbar_257,delta_mbar_257_minus_129,physical_129,physical_257
4,NGC 3379,inner,26.932691,26.917244,-0.015447,True,True
12,NGC 3379,outer,27.100979,27.085429,-0.015550,True,True
20,NGC 1380,inner,28.092690,28.077243,-0.015447,True,True
28,NGC 1380,outer,28.138536,28.123068,-0.015468,True,True
36,NGC 4486,inner,27.919839,27.904295,-0.015543,True,True
44,NGC 4486,outer,27.851107,27.835598,-0.015510,True,True


### PSF stamp-size sensitivity (English figure set)

The left panel compares the encircled-energy curves of the unit-integral
129- and 257-pixel STPSF models. The right panel gives the change in the
fitted SBF magnitude when the full windowed FFT fit is repeated with the
257-pixel model. The test therefore checks both the omitted PSF wings and
their effect over the adopted frequency range. The production size is
**129 pixels** (not 127 pixels).


In [22]:
for state in states:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    for stamp_size, group in state.psf_encircled_energy.groupby(
        "psf_size_pix"
    ):
        axes[0].plot(
            group["radius_pix"], group["encircled_energy"],
            marker="o", label=f"{stamp_size} pixels",
        )
    axes[0].set(
        xlabel="Radius (pixels)",
        ylabel="Encircled-energy fraction",
        xscale="log",
        title="Unit-integral PSF models",
    )
    axes[0].legend(frameon=False)

    classic = state.psf_size_comparison[
        state.psf_size_comparison["noise_model"].eq("constant")
    ]
    for ring, marker, label in [
        ("inner", "o", "Inner annulus"),
        ("outer", "s", "Outer annulus"),
    ]:
        group = (
            classic[classic["ring"].eq(ring)]
            .set_index("variant")
            .loc[[item[0] for item in WINSOR_VARIANTS]]
        )
        axes[1].plot(
            np.arange(len(group)),
            group["delta_mbar_257_minus_129"],
            marker=marker, label=label,
        )
    axes[1].axhline(0, color="0.4", lw=1)
    axes[1].set_xticks(np.arange(4), ["raw", "3.0", "3.5", "4.0"])
    axes[1].set(
        xlabel="Winsorization threshold",
        ylabel=r"$\bar m_{257}-\bar m_{129}$ (mag)",
        title="Repeated windowed FFT fit",
    )
    axes[1].legend(frameon=False)

    fig.suptitle(f"{state.galaxy}: PSF stamp-size sensitivity")
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    fig.savefig(
        state.figure_dir
        / f"{state.galaxy_tag}_psf_129_257_sensitivity_english.pdf",
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)


/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/4132819731.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/4132819731.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/4132819731.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



## 8. Общая таблица трёх галактик

Итоговая таблица содержит ровно шесть строк: три галактики × два кольца.
Она собирается непосредственно из общих DataFrame, без повторного поиска
CSV на диске. Здесь оставлены только величины, необходимые для научного
решения; подробные параметры доступны в таблицах предыдущих этапов.


In [23]:

pilot_rows = []
score_by_galaxy = galaxies.set_index("galaxy")["visual_score"]
mode_by_galaxy = galaxies.set_index("galaxy")["noise_tile_mode"]

for galaxy in galaxies["galaxy"]:
    target_fit = fit_summary_df[fit_summary_df["galaxy"].eq(galaxy)]
    target_winsor = winsor_by_ring_df[winsor_by_ring_df["galaxy"].eq(galaxy)]
    target_model = model_comparison_df[model_comparison_df["galaxy"].eq(galaxy)]
    target_residual = fit_residuals_df[fit_residuals_df["galaxy"].eq(galaxy)]
    target_psf = psf_comparison_df[psf_comparison_df["galaxy"].eq(galaxy)]

    for ring in ["inner", "outer"]:
        classic = target_fit[
            target_fit["ring"].eq(ring)
            & target_fit["noise_model"].eq("constant")
        ].set_index("variant")
        adopted = classic.loc["winsor_3.5"]
        raw = classic.loc["raw"]
        nk = target_fit[
            target_fit["ring"].eq(ring)
            & target_fit["variant"].eq("winsor_3.5")
            & target_fit["noise_model"].eq("N(k)")
        ].iloc[0]
        model_row = target_model[
            target_model["ring"].eq(ring)
            & target_model["variant"].eq("winsor_3.5")
        ].iloc[0]
        changed = target_winsor[
            target_winsor["ring"].eq(ring)
            & target_winsor["variant"].eq("winsor_3.5")
        ].iloc[0]
        psf_row = target_psf[
            target_psf["ring"].eq(ring)
            & target_psf["variant"].eq("winsor_3.5")
            & target_psf["noise_model"].eq("constant")
        ].iloc[0]

        classic_residual = target_residual[
            target_residual["ring"].eq(ring)
            & target_residual["variant"].eq("winsor_3.5")
            & target_residual["noise_model"].eq("constant")
        ]["residual_sigma"].to_numpy(float)
        nk_residual = target_residual[
            target_residual["ring"].eq(ring)
            & target_residual["variant"].eq("winsor_3.5")
            & target_residual["noise_model"].eq("N(k)")
        ]["residual_sigma"].to_numpy(float)
        mbar_values = classic["mbar_observed"].to_numpy(float)
        P0_values = classic["P0"].to_numpy(float)

        pilot_rows.append({
            "galaxy": galaxy,
            "visual_score": score_by_galaxy[galaxy],
            "ring": ring,
            "noise_tile_mode": mode_by_galaxy[galaxy],
            "mbar_adopted_3p5": adopted["mbar_observed"],
            "mbar_raw": raw["mbar_observed"],
            "delta_mbar_3p5_minus_raw": adopted["mbar_observed"] - raw["mbar_observed"],
            "winsor_mbar_span": float(np.nanmax(mbar_values) - np.nanmin(mbar_values)),
            "changed_fraction_3p5": changed["changed_fraction"],
            "P0_fractional_span": float(
                (np.nanmax(P0_values) - np.nanmin(P0_values)) / np.nanmedian(P0_values)
            ),
            "delta_aicc_Nk_minus_constant": model_row["delta_aicc_Nk_minus_constant"],
            "Nk_physical_solution_fraction": nk["physical_solution_fraction"],
            "classic_residual_rms_sigma": float(np.sqrt(np.nanmean(classic_residual**2))),
            "Nk_residual_rms_sigma": float(np.sqrt(np.nanmean(nk_residual**2))),
            "delta_mbar_257_minus_129": psf_row["delta_mbar_257_minus_129"],
        })

pilot_summary_df = pd.DataFrame(pilot_rows)
if len(pilot_summary_df) != 2 * len(galaxies):
    raise RuntimeError("Итоговая таблица должна содержать шесть строк")
pilot_summary_df.to_csv(
    pilot_table_dir / "sbf2_three_galaxy_systematics_pilot.csv", index=False
)
display(pilot_summary_df[[
    "galaxy", "ring", "mbar_adopted_3p5", "mbar_raw",
    "delta_mbar_3p5_minus_raw", "winsor_mbar_span",
    "changed_fraction_3p5", "delta_aicc_Nk_minus_constant",
    "Nk_physical_solution_fraction", "delta_mbar_257_minus_129",
]])


,galaxy,ring,mbar_adopted_3p5,mbar_raw,delta_mbar_3p5_minus_raw,winsor_mbar_span,changed_fraction_3p5,delta_aicc_Nk_minus_constant,Nk_physical_solution_fraction,delta_mbar_257_minus_129
0,NGC 3379,inner,26.933681,26.809381,0.124300,0.265782,0.043143,17027.353797,0.2,-0.015447
1,NGC 3379,outer,27.101682,27.096186,0.005496,0.023075,0.001848,31478.319345,0.0,-0.015550
2,NGC 1380,inner,28.094008,28.029838,0.064170,0.150753,0.022739,16335.143355,0.4,-0.015447
3,NGC 1380,outer,28.139850,28.133785,0.006065,0.021619,0.001984,46926.207103,0.6,-0.015468
4,NGC 4486,inner,27.920484,27.781252,0.139233,0.299721,0.059640,226.075802,0.0,-0.015543
5,NGC 4486,outer,27.851802,27.847569,0.004233,0.023457,0.002227,-226.762197,0.0,-0.015510


### Как читать итоговый рисунок

Четыре панели сводят сдвиг относительно raw, полный размах по порогам,
сравнение моделей шума и размер PSF. Порог 0.03 mag — ориентир
практически значимой систематики. Знак ΔAICc трактуется только вместе с
физичностью $P_0$: отрицательное значение при $P_0 ≤ P_r$ не является
подтверждением модели $N(k)$.

In [24]:

labels = (
    pilot_summary_df["galaxy"].str.replace("NGC ", "", regex=False)
    + " " + pilot_summary_df["ring"].map({"inner": "внутр.", "outer": "внеш."})
)
x = np.arange(len(pilot_summary_df))
fig, axes = plt.subplots(2, 2, figsize=(11, 7))

axes[0, 0].bar(x, pilot_summary_df["delta_mbar_3p5_minus_raw"])
axes[0, 0].axhline(0, color="0.3", lw=1)
axes[0, 0].set_ylabel(r"$\bar m_{3.5}-\bar m_{raw}$ (mag)")
axes[0, 0].set_title("Сдвиг от принятого винзорирования")

axes[0, 1].bar(x, pilot_summary_df["winsor_mbar_span"])
axes[0, 1].axhline(0.03, color="#dc2626", ls="--", lw=1)
axes[0, 1].set_ylabel("Полный размах (mag)")
axes[0, 1].set_title("Чувствительность к порогу")

physical = pilot_summary_df["Nk_physical_solution_fraction"].eq(1)
bars = axes[1, 0].bar(
    x, pilot_summary_df["delta_aicc_Nk_minus_constant"],
    color=np.where(physical, "#2563eb", "0.65"),
)
for bar, ok in zip(bars, physical):
    if not ok:
        bar.set_hatch("///")
axes[1, 0].axhline(0, color="0.3", lw=1)
axes[1, 0].set_yscale("symlog", linthresh=1)
axes[1, 0].set_ylabel(r"$\Delta$AICc: $N(k)-$константа")
axes[1, 0].set_title("Штриховка: нефизическое решение")

axes[1, 1].bar(x, pilot_summary_df["delta_mbar_257_minus_129"])
axes[1, 1].axhline(0, color="0.3", lw=1)
axes[1, 1].axhline(0.03, color="#dc2626", ls="--", lw=1)
axes[1, 1].axhline(-0.03, color="#dc2626", ls="--", lw=1)
axes[1, 1].set_ylabel(r"$\bar m_{257}-\bar m_{129}$ (mag)")
axes[1, 1].set_title("Размер штампа PSF")

for axis in axes.ravel():
    axis.set_xticks(x, labels, rotation=45, ha="right")
fig.tight_layout()
fig.savefig(
    pilot_figure_dir / "sbf2_three_galaxy_systematics_pilot.pdf",
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

checks = [
    ("Все 6 production-измерений воспроизведены", bool(closure_df["passed"].all()), "обязательно"),
    ("Размах винзорирования везде < 0.03 mag",
     bool((pilot_summary_df["winsor_mbar_span"] <= 0.03).all()), "0.03 mag"),
    ("Сдвиг PSF везде < 0.03 mag",
     bool((pilot_summary_df["delta_mbar_257_minus_129"].abs() <= 0.03).all()), "0.03 mag"),
    ("Все N(k)-решения физичны",
     bool((pilot_summary_df["Nk_physical_solution_fraction"] == 1).all()), "доля = 1"),
]
pilot_decision_df = pd.DataFrame(
    checks, columns=["check", "passed", "criterion"]
)
pilot_decision_df.to_csv(
    pilot_table_dir / "sbf2_three_galaxy_systematics_decision.csv", index=False
)
display(pilot_decision_df)


/var/folders/d4/pdhkc1y53l5bywltpcg3j0cm0000gp/T/ipykernel_70527/1751025224.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,check,passed,criterion
0,Все 6 production-измерений воспроизведены,True,обязательно
1,Размах винзорирования везде < 0.03 mag,False,0.03 mag
2,Сдвиг PSF везде < 0.03 mag,True,0.03 mag
3,Все N(k)-решения физичны,False,доля = 1


## 9. Сохранённые продукты

Короткая карта папок для всех трёх галактик.

In [25]:

artifacts_df = pd.DataFrame([
    {
        "galaxy": state.galaxy,
        "tables": str(state.table_dir.relative_to(project_root)),
        "figures": str(state.figure_dir.relative_to(project_root)),
    }
    for state in states
])
display(artifacts_df)


,galaxy,tables,figures
0,NGC 3379,runs/sbf2_systematics/NGC_3379/tables,runs/sbf2_systematics/NGC_3379/figures
1,NGC 1380,runs/sbf2_systematics/NGC_1380/tables,runs/sbf2_systematics/NGC_1380/figures
2,NGC 4486,runs/sbf2_systematics/NGC_4486/tables,runs/sbf2_systematics/NGC_4486/figures
